# Laboratorio 4 ML — Asistente de compras por voz (ESCI + RAG + agente + TTS clonado)

Asistente que recibe consultas habladas en español (ASR), recupera productos del catálogo ESCI (RAG), ejecuta
operaciones sobre una tienda simulada mediante un agente con herramientas y responde por voz con una voz clonada (TTS).

**Todo el sistema está en este notebook** (tablero de control, datos, tienda, RAG, agente, voz, escenarios y evaluación).
**Parámetros:** celda *Tablero de control* (§1). Edítala, reinicia el kernel y ejecuta de arriba a abajo.
Lo ya ejecutado se omite (`output/resultados/registro.json`).

**Carpetas:** `input/` (data ESCI, voz de referencia y audios de los escenarios) · `output/` (datos, índice, tienda,
escenarios, resultados). Instalación: `requirements.txt`.

**Secciones:** 0 Enunciado · 1 Configuración y entorno · 2 Datos · 3 Tienda · 4 Índice RAG · 5 Evaluación de la
recuperación (modelo base y ablación) · 6 Modelos de lenguaje y voz · 7 Agente y autoprueba · 8 Escenarios por texto ·
9 Escenarios por voz y repetición con transcripción correcta · 10 Métricas · 11 Gráficos · 12 Análisis de aciertos y
fallos · 13 Demo interactiva · 14 Instrucciones, versiones y diferencias · 15 Aplicativo de prueba

## 0. Enunciado
*(pegar aquí el enunciado del Laboratorio 4)*

## 1. Tablero de control, bitácora y entorno

In [1]:
# ===== TABLERO DE CONTROL: edita solo esta celda, reinicia el kernel y ejecuta todo =====
# Cada paso guarda un hash de su configuracion: se repite solo lo que cambio o falta.
from pathlib import Path

# Raiz del proyecto = carpeta donde esta este archivo (igual en Windows, Linux y Colab)
RAIZ = Path.cwd()   # carpeta donde esta este notebook

# ===== 1. RUTAS =====
RUTA_INPUT = RAIZ / "input"                         # data ESCI descargada (solo lectura)
RUTA_MULTIMEDIA = RAIZ / "multimedia"               # voces grabadas con multimedia.bat (multimedia001, 002...)
_ACTIVA = RUTA_MULTIMEDIA / "ACTIVA.txt"            # voz elegida en el grabador ("Usar en el programa")
VOZ_ID = _ACTIVA.read_text().strip() if _ACTIVA.exists() else "multimedia001"   # o fijarla a mano
RUTA_VOZ = RUTA_MULTIMEDIA / VOZ_ID
RUTA_VOZ_REF = RUTA_VOZ / "referencia.wav"          # voz propia o autorizada para clonar
RUTA_AUDIOS = RUTA_VOZ / "audios"                   # audios de los escenarios por voz (E01_t1.wav ...)
TEXTO_REF = ("Hola, esta es mi voz de referencia para el asistente de compras. "
             "La grabo en un lugar tranquilo, hablando con calma y a un ritmo natural.")  # lo que se lee en referencia.wav
RUTA_OUTPUT = RAIZ / "output"                       # todo lo generado
RUTA_DATOS = RUTA_OUTPUT / "datos"                  # catalogo y consultas
RUTA_EVAL = RUTA_DATOS / "evaluacion"               # etiquetas ESCI: solo evaluacion, el agente nunca las lee
RUTA_INDICE = RUTA_OUTPUT / "indice"                # embeddings del catalogo
RUTA_TIENDA = RUTA_OUTPUT / "tienda"                # base SQLite (base restaurable + copia de trabajo)
RUTA_ESC = RUTA_OUTPUT / "escenarios"               # registros, audios y estado final por escenario
RUTA_RES = RUTA_OUTPUT / "resultados"               # bitacora, metricas, figuras

# ===== 2. DATOS (ESCI small, locale es) =====
SEMILLA = 2026
N_CONSULTAS_EVAL = 250      # split test (enunciado: >= 200)
N_CONSULTAS_DEV = 150       # split train: desarrollo y ajuste
MIN_CATALOGO = 5000         # enunciado: >= 5000 productos (se completa con distractores si falta)
MAX_CHARS_TEXTO = 600       # recorte de descripcion y caracteristicas al indexar

# ===== 3. TIENDA SIMULADA =====
PRECIO_LOGNORMAL = (3.0, 0.8)   # log-precio ~ N(mu, sigma): mediana ~20 euros
PRECIO_RANGO = (1.99, 399.99)
PROB_AGOTADO = 0.08
STOCK_MAX = 25
CANTIDAD_MAX = 20               # unidades por producto en un pedido

# ===== 4. RECUPERACION (RAG) =====
MODELO_EMB = "intfloat/multilingual-e5-base"
MODELO_RERANK = "BAAI/bge-reranker-v2-m3"
TOP_CANDIDATOS = 50             # top de BM25 y de denso antes de fusionar
TOP_RERANK = 30                 # candidatos que pasan al reranker
GRID = [i / 10 for i in range(11)]   # alpha (denso vs BM25) y beta (reranker vs hibrido), ajustados en dev
K_NDCG = 10
GANANCIAS = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}   # enunciado

# ===== 5. AGENTE (LLM local) =====
MODELO_LLM = "Qwen/Qwen2.5-7B-Instruct"
LLM_4BIT = True                 # nf4: ~5.5 GB de VRAM
LLM_MAX_TOKENS = 384
MAX_PASOS_AGENTE = 8            # llamadas al LLM por turno
TURNOS_HISTORIAL = 6
K_RESULTADOS = 5                # productos que devuelve buscar_productos

# ===== 6. VOZ =====
MODELO_ASR = "openai/whisper-large-v3-turbo"
MODELO_TTS = "tts_models/multilingual/multi-dataset/xtts_v2"
MODELO_SEMEJANZA = "microsoft/wavlm-base-plus-sv"
SR_ASR = 16000
SEGUNDOS_GRABACION = 7
TTS_EN_ESCENARIOS_TEXTO = False  # True: tambien sintetiza voz en los 14 escenarios de texto (mas lento)
DISPOSITIVO = "auto"              # RTX 5080 requiere una build de PyTorch compatible con sm_120

# ===== 7. EJECUCION =====
EJECUTAR_EVAL_RECUPERACION = True
EJECUTAR_ESCENARIOS_TEXTO = True
EJECUTAR_ESCENARIOS_VOZ = False   # requiere la voz activa de multimedia/ completa (multimedia.bat)
MODO_INTERACTIVO = False         # True: habilita la demo con microfono (no usar con nbconvert)
FORZAR_REEJECUTAR = False        # True: ignora lo ya ejecutado (luego volver a False)

In [2]:
# ===== Bitacora de pasos y registro de lo ya ejecutado =====
"""Bitacora de pasos (CSV + MD), progreso en vivo y registro de lo ya ejecutado (hash de configuracion)."""
import csv
import datetime as dt
import hashlib
import json
import time
from contextlib import contextmanager
from pathlib import Path

COLS_PASOS = ["ejecucion", "paso", "inicio", "fin", "duracion_s", "tiempo_usado", "avance", "estado", "comentario"]


def hms(seg):
    h, r = divmod(seg, 3600)
    m, s = divmod(r, 60)
    return f"{int(h):02d}:{int(m):02d}:{s:06.3f}"


def hash_cfg(cfg):
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:12]


class Bitacora:
    def __init__(self, carpeta, forzar=False):
        self.carpeta = Path(carpeta)
        self.carpeta.mkdir(parents=True, exist_ok=True)
        self.forzar = forzar
        self.ej = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
        self.csv = self.carpeta / "bitacora_pasos.csv"
        self.md = self.carpeta / "bitacora_pasos.md"
        self.progreso = self.carpeta / "progreso.md"
        self.reg = self.carpeta / "registro.json"
        self.filas = list(csv.DictReader(open(self.csv, encoding="utf-8"))) if self.csv.exists() else []
        self.actual = "-"

    # --- registro de lo ya ejecutado ---
    def leer_registro(self):
        return json.loads(self.reg.read_text(encoding="utf-8")) if self.reg.exists() else {}

    def registrar(self, clave, info):
        r = self.leer_registro()
        r[clave] = info
        self.reg.write_text(json.dumps(r, indent=2, ensure_ascii=False), encoding="utf-8")

    def ya_ejecutado(self, clave, h, salidas=()):
        e = self.leer_registro().get(clave)
        return (not self.forzar and e is not None and e.get("hash") == h
                and all(Path(s).exists() for s in salidas))

    # --- tabla de pasos ---
    def _guardar(self):
        with open(self.csv, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, COLS_PASOS, extrasaction="ignore")
            w.writeheader()
            w.writerows(self.filas)
        lin = ["| " + " | ".join(COLS_PASOS) + " |", "|" + "---|" * len(COLS_PASOS)]
        lin += ["| " + " | ".join(str(r.get(c, "")).replace("|", "/") for c in COLS_PASOS) + " |" for r in self.filas]
        self.md.write_text("\n".join(lin) + "\n", encoding="utf-8")
        mios = [r for r in self.filas if r["ejecucion"] == self.ej]
        hechos = sum(1 for r in mios if str(r["avance"]) == "1")
        self.progreso.write_text(f"# Progreso (ejecucion {self.ej})\n\nPasos completados: {hechos} de {len(mios)}\n"
                                 f"Paso en curso: {self.actual}\n", encoding="utf-8")

    @contextmanager
    def paso(self, nombre, comentario=""):
        info = {"estado": "OK", "comentario": comentario}
        t0 = time.time()
        fila = {"ejecucion": self.ej, "paso": nombre, "inicio": dt.datetime.now().strftime("%H:%M:%S"),
                "fin": "", "duracion_s": "", "tiempo_usado": "", "avance": 0, "estado": "EN CURSO",
                "comentario": comentario}
        self.filas.append(fila)
        self.actual = nombre
        self._guardar()
        print(f"> {nombre}")
        try:
            yield info
        except Exception as e:
            info["estado"] = "ERROR"
            info["comentario"] = f"{info['comentario']} | {type(e).__name__}: {e}"
            raise
        finally:
            seg = time.time() - t0
            fila.update({"fin": dt.datetime.now().strftime("%H:%M:%S"), "duracion_s": f"{seg:.3f}",
                         "tiempo_usado": hms(seg), "avance": 1 if info["estado"] in ("OK", "OMITIDO") else 0,
                         "estado": info["estado"], "comentario": info["comentario"]})
            self.actual = "-"
            self._guardar()
            print(f"  {info['estado']} | {hms(seg)}")

    def ejecutar(self, clave, cfg, salidas, fn):
        """Ejecuta fn() salvo que ya se haya ejecutado con la misma configuracion y existan las salidas."""
        h = hash_cfg(cfg)
        if self.ya_ejecutado(clave, h, salidas):
            with self.paso(clave) as info:
                info["estado"], info["comentario"] = "OMITIDO", f"ya ejecutado (hash {h})"
            return h
        with self.paso(clave, f"hash {h}"):
            fn()
            faltan = [str(s) for s in salidas if not Path(s).exists()]
            if faltan:
                raise RuntimeError(f"salidas faltantes: {faltan}")
            self.registrar(clave, {"hash": h, "fecha": dt.datetime.now().strftime("%Y-%m-%d %H:%M")})
        return h

In [3]:
import importlib, json, platform, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
C = sys.modules[__name__]   # acceso uniforme a los parametros del tablero: C.NOMBRE

for r in (C.RUTA_DATOS, C.RUTA_EVAL, C.RUTA_INDICE, C.RUTA_TIENDA, C.RUTA_ESC, C.RUTA_RES):
    r.mkdir(parents=True, exist_ok=True)
FIG = C.RUTA_RES / "figuras"; FIG.mkdir(exist_ok=True)
BIT = Bitacora(C.RUTA_RES, C.FORZAR_REEJECUTAR)

def guardar_fig(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=150, bbox_inches="tight"); print("Figura:", FIG / f"{nombre}.png")

try:
    import torch
    CUDA = torch.cuda.is_available()
    GPU = f"{torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB)" if CUDA else "sin GPU"
except ImportError:
    torch, CUDA, GPU = None, False, "torch no instalado"
DISP = ("cuda" if CUDA else "cpu") if C.DISPOSITIVO == "auto" else C.DISPOSITIVO

def vram():
    return round(torch.cuda.memory_allocated() / 2**30, 2) if CUDA else 0.0

VERSIONES = {"python": sys.version.split()[0], "sistema": platform.platform(), "gpu": GPU}
for m in ("torch", "transformers", "sentence_transformers", "accelerate", "bitsandbytes", "TTS", "rank_bm25", "jiwer", "pandas", "numpy"):
    try:
        VERSIONES[m] = importlib.import_module(m).__version__
    except Exception as e:
        VERSIONES[m] = f"no disponible ({type(e).__name__})"
(C.RUTA_RES / "versiones.json").write_text(json.dumps(VERSIONES, indent=2), encoding="utf-8")
pd.Series(VERSIONES).to_frame("version")

/home/winzcode/proyectos/rainfall-runoff-forecasting/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


,version
python,3.11.9
sistema,Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-...
gpu,NVIDIA GeForce RTX 5070 Ti Laptop GPU (11.9 GB)
torch,2.14.0+cu130
transformers,4.51.3
sentence_transformers,4.1.0
accelerate,1.6.0
bitsandbytes,0.50.2
TTS,0.26.0
rank_bm25,no disponible (AttributeError)


## 2. Datos: catálogo, consultas y etiquetas
- ESCI **small** filtrado por **español**. Consultas separadas por `query_id`: **eval** = split *test* (≥ 200), **dev** = split *train*.
- Catálogo = todos los productos anotados de esas consultas (+ distractores si faltara para llegar a 5 000).
- Las etiquetas ESCI se guardan aparte en `output/datos/evaluacion/qrels.parquet` y **solo** las usa la evaluación;
  el agente únicamente accede a `catalogo.parquet` y a la tienda.

In [4]:
# ===== Datos: catalogo, consultas y etiquetas =====
"""Catalogo, consultas (dev = train, eval = test) y etiquetas ESCI a partir de la version small en espanol."""
import html
import re

import numpy as np
import pandas as pd

ARCH_EX = "shopping_queries_dataset_examples.parquet"
ARCH_PR = "shopping_queries_dataset_products.parquet"
COLS_CATALOGO = {"product_id": "product_id", "product_title": "titulo", "product_description": "descripcion",
        "product_bullet_point": "caracteristicas", "product_brand": "marca", "product_color": "color"}


def limpiar(t):
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return ""
    t = re.sub(r"<[^>]+>", " ", html.unescape(str(t)))
    return re.sub(r"\s+", " ", t).strip()


def preparar(ruta_input, ruta_datos, ruta_eval, n_eval, n_dev, min_catalogo, semilla):
    """Genera catalogo.parquet, consultas.parquet y evaluacion/qrels.parquet. Devuelve un resumen."""
    ex = pd.read_parquet(ruta_input / ARCH_EX, columns=["query_id", "query", "product_id", "esci_label",
                                                        "small_version", "product_locale", "split"])
    ex = ex[(ex["small_version"] == 1) & (ex["product_locale"] == "es")]
    rng = np.random.default_rng(semilla)
    q_test = np.sort(ex.loc[ex["split"] == "test", "query_id"].unique())
    q_train = np.sort(ex.loc[ex["split"] == "train", "query_id"].unique())
    q_eval = np.sort(rng.choice(q_test, n_eval, replace=False))
    q_dev = np.sort(rng.choice(q_train, n_dev, replace=False))
    sel = ex[ex["query_id"].isin(np.r_[q_eval, q_dev])]

    pr = pd.read_parquet(ruta_input / ARCH_PR, columns=list(COLS_CATALOGO) + ["product_locale"],
                         filters=[("product_locale", "==", "es")]).drop(columns="product_locale")
    pr = pr.drop_duplicates("product_id").set_index("product_id")
    ids = set(sel["product_id"])
    faltan = max(0, min_catalogo - len(ids))
    if faltan:   # distractores: productos ES fuera de las consultas elegidas
        resto = np.sort(np.array(sorted(set(pr.index) - ids)))
        ids |= set(rng.choice(resto, faltan, replace=False))
    cat = pr.loc[sorted(ids)].reset_index().rename(columns=COLS_CATALOGO)
    for c in COLS_CATALOGO.values():
        if c != "product_id":
            cat[c] = cat[c].map(limpiar)

    consultas = (sel.drop_duplicates("query_id")[["query_id", "query", "split"]]
                 .assign(conjunto=lambda d: np.where(d["split"] == "test", "eval", "dev"))
                 .drop(columns="split").sort_values("query_id").reset_index(drop=True))
    qrels = sel[["query_id", "product_id", "esci_label"]].reset_index(drop=True)
    assert qrels["product_id"].isin(cat["product_id"]).all(), "producto anotado fuera del catalogo"
    assert len(cat) >= min_catalogo and (consultas["conjunto"] == "eval").sum() >= 200

    ruta_datos.mkdir(parents=True, exist_ok=True)
    ruta_eval.mkdir(parents=True, exist_ok=True)
    cat.to_parquet(ruta_datos / "catalogo.parquet", index=False)
    consultas.to_parquet(ruta_datos / "consultas.parquet", index=False)
    qrels.to_parquet(ruta_eval / "qrels.parquet", index=False)

    vc = qrels.merge(consultas, on="query_id").groupby(["conjunto", "esci_label"]).size().unstack(fill_value=0)
    return {"productos": len(cat), "distractores": faltan,
            "consultas_eval": int((consultas["conjunto"] == "eval").sum()),
            "consultas_dev": int((consultas["conjunto"] == "dev").sum()),
            "pares_anotados": len(qrels), "etiquetas": vc.to_dict(orient="index"),
            "faltantes_%": {c: round(float((cat[c] == "").mean() * 100), 1)
                            for c in ["descripcion", "caracteristicas", "marca", "color"]}}

In [5]:
CFG_DATOS = {"n_eval": C.N_CONSULTAS_EVAL, "n_dev": C.N_CONSULTAS_DEV, "min": C.MIN_CATALOGO, "semilla": C.SEMILLA}
H_DATOS = BIT.ejecutar("2. Datos (catalogo, consultas, qrels)", CFG_DATOS,
                       [C.RUTA_DATOS / "catalogo.parquet", C.RUTA_DATOS / "consultas.parquet", C.RUTA_EVAL / "qrels.parquet"],
                       lambda: (C.RUTA_RES / "resumen_datos.json").write_text(json.dumps(
                           preparar(C.RUTA_INPUT, C.RUTA_DATOS, C.RUTA_EVAL, C.N_CONSULTAS_EVAL, C.N_CONSULTAS_DEV,
                                    C.MIN_CATALOGO, C.SEMILLA), indent=1, ensure_ascii=False), encoding="utf-8"))
CAT = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
CONS = pd.read_parquet(C.RUTA_DATOS / "consultas.parquet")
QRELS = pd.read_parquet(C.RUTA_EVAL / "qrels.parquet")
assert "esci_label" not in CAT.columns, "el catalogo no debe contener etiquetas"
print(json.loads((C.RUTA_RES / "resumen_datos.json").read_text(encoding="utf-8")))
CAT.head(3)

> 2. Datos (catalogo, consultas, qrels)
  OMITIDO | 00:00:00.002
{'productos': 10688, 'distractores': 0, 'consultas_eval': 250, 'consultas_dev': 150, 'pares_anotados': 10928, 'etiquetas': {'dev': {'C': 305, 'E': 1645, 'I': 690, 'S': 1399}, 'eval': {'C': 548, 'E': 2814, 'I': 1032, 'S': 2495}}, 'faltantes_%': {'descripcion': 47.9, 'caracteristicas': 12.0, 'marca': 4.7, 'color': 37.2}}


,product_id,titulo,descripcion,caracteristicas,marca,color
0,0060824972,The Thief: 1 (Queen's Thief),,,Greenwillow Books,
1,0142501123,ROOM ON THE BROOM (Picture Puffins),,,Puffin Books,Multicolor
2,0190628081,Through The Looking Glass: John Cage and Avant...,,,OUP USA,


## 3. Tienda simulada (semilla fija y restaurable)
Precios (log-normal, terminados en ,99) y existencias (8 % agotados) generados con `SEMILLA`. La base queda en
`tienda_base.sqlite`; `restaurar()` la copia sobre `tienda.sqlite`. La huella (hash de inventario y pedidos) demuestra
que dos restauraciones dan el mismo estado.

In [6]:
# ===== Tienda simulada (SQLite) =====
"""Tienda simulada en SQLite: precios, existencias, carritos, cotizaciones y pedidos.

- Datos ficticios generados con semilla fija en una base "base"; restaurar() copia la base sobre la de trabajo.
- Montos en centimos (enteros) para evitar errores de redondeo.
- confirmar() es atomico (BEGIN IMMEDIATE): o se registra todo (pedido + inventario) o nada.
- Un pedido por cotizacion (UNIQUE): repetir la confirmacion devuelve el mismo pedido, sin duplicar.
"""
import hashlib
import json
import shutil
import sqlite3

import numpy as np

ESQUEMA = """
CREATE TABLE productos(product_id TEXT PRIMARY KEY, precio_cent INTEGER NOT NULL CHECK(precio_cent > 0),
                       stock INTEGER NOT NULL CHECK(stock >= 0));
CREATE TABLE carritos(id INTEGER PRIMARY KEY AUTOINCREMENT, estado TEXT NOT NULL DEFAULT 'abierto',
                      version INTEGER NOT NULL DEFAULT 0);
CREATE TABLE carrito_items(carrito_id INTEGER NOT NULL, product_id TEXT NOT NULL,
                           cantidad INTEGER NOT NULL CHECK(cantidad > 0), PRIMARY KEY(carrito_id, product_id));
CREATE TABLE cotizaciones(id TEXT PRIMARY KEY, carrito_id INTEGER, version INTEGER, total_cent INTEGER,
                          presupuesto_cent INTEGER, items TEXT, valida INTEGER);
CREATE TABLE pedidos(id INTEGER PRIMARY KEY AUTOINCREMENT, cotizacion_id TEXT UNIQUE NOT NULL,
                     carrito_id INTEGER, total_cent INTEGER);
CREATE TABLE pedido_items(pedido_id INTEGER, product_id TEXT, cantidad INTEGER, precio_cent INTEGER);
"""


def euros(cent):
    return None if cent is None else round(cent / 100, 2)


class ErrorTienda(Exception):
    pass


class Tienda:
    def __init__(self, ruta_base, ruta_trabajo, cantidad_max):
        self.base, self.trabajo, self.cmax = ruta_base, ruta_trabajo, cantidad_max
        self.con = None

    # --- creacion y restauracion ---
    @staticmethod
    def crear_base(ruta_base, product_ids, semilla, mu, sigma, rango, prob_agotado, stock_max):
        rng = np.random.default_rng(semilla)
        ids = sorted(product_ids)
        precio = np.clip(np.exp(rng.normal(mu, sigma, len(ids))), *rango)
        cent = (np.floor(precio) * 100 + 99).astype(int)                       # precios x,99
        stock = np.where(rng.random(len(ids)) < prob_agotado, 0, rng.integers(1, stock_max + 1, len(ids)))
        ruta_base.parent.mkdir(parents=True, exist_ok=True)
        ruta_base.unlink(missing_ok=True)
        con = sqlite3.connect(ruta_base)
        con.executescript(ESQUEMA)
        con.executemany("INSERT INTO productos VALUES (?,?,?)",
                        zip(ids, map(int, cent), map(int, stock)))
        con.commit()
        con.close()

    def restaurar(self):
        if self.con is not None:
            self.con.close()
        shutil.copyfile(self.base, self.trabajo)
        self.con = sqlite3.connect(self.trabajo, isolation_level=None)   # transacciones explicitas
        self.con.execute("PRAGMA foreign_keys=ON")

    def huella(self):
        """Hash del inventario y pedidos: dos restauraciones deben dar la misma huella."""
        filas = self.con.execute("SELECT * FROM productos ORDER BY product_id").fetchall()
        filas += self.con.execute("SELECT * FROM pedidos ORDER BY id").fetchall()
        return hashlib.sha256(json.dumps(filas).encode()).hexdigest()[:16]

    # --- consultas ---
    def producto(self, pid):
        r = self.con.execute("SELECT precio_cent, stock FROM productos WHERE product_id=?", (pid,)).fetchone()
        return None if r is None else {"precio_cent": r[0], "stock": r[1]}

    def precios_stock(self, pids):
        q = f"SELECT product_id, precio_cent, stock FROM productos WHERE product_id IN ({','.join('?' * len(pids))})"
        return {p: {"precio_cent": c, "stock": s} for p, c, s in self.con.execute(q, list(pids))}

    # --- carrito ---
    def nuevo_carrito(self):
        return self.con.execute("INSERT INTO carritos DEFAULT VALUES").lastrowid

    def carrito(self, cid):
        est = self.con.execute("SELECT estado, version FROM carritos WHERE id=?", (cid,)).fetchone()
        if est is None:
            raise ErrorTienda("carrito inexistente")
        items = [{"product_id": p, "cantidad": c, "precio_cent": pc, "stock": s, "subtotal_cent": c * pc}
                 for p, c, pc, s in self.con.execute(
                     "SELECT i.product_id, i.cantidad, p.precio_cent, p.stock FROM carrito_items i "
                     "JOIN productos p USING(product_id) WHERE i.carrito_id=? ORDER BY i.rowid", (cid,))]
        return {"carrito_id": cid, "estado": est[0], "version": est[1], "items": items,
                "total_cent": sum(i["subtotal_cent"] for i in items)}

    def fijar_cantidad(self, cid, pid, cantidad):
        """cantidad = 0 elimina la linea. Valida entero, limite, stock y carrito abierto."""
        if not isinstance(cantidad, int) or isinstance(cantidad, bool) or cantidad < 0:
            raise ErrorTienda("cantidad invalida: debe ser un entero mayor o igual a 0")
        if cantidad > self.cmax:
            raise ErrorTienda(f"cantidad maxima por producto: {self.cmax}")
        prod = self.producto(pid)
        if prod is None:
            raise ErrorTienda("producto inexistente")
        if self.carrito(cid)["estado"] != "abierto":
            raise ErrorTienda("el carrito ya fue cerrado por un pedido")
        if cantidad > prod["stock"]:
            raise ErrorTienda(f"stock insuficiente: disponibles {prod['stock']}")
        self.con.execute("BEGIN")
        if cantidad == 0:
            self.con.execute("DELETE FROM carrito_items WHERE carrito_id=? AND product_id=?", (cid, pid))
        else:
            self.con.execute("INSERT INTO carrito_items VALUES (?,?,?) ON CONFLICT(carrito_id, product_id) "
                             "DO UPDATE SET cantidad=excluded.cantidad", (cid, pid, cantidad))
        self.con.execute("UPDATE carritos SET version=version+1 WHERE id=?", (cid,))
        self.con.execute("COMMIT")
        return self.carrito(cid)

    # --- cotizacion y pedido ---
    def cotizar(self, cid, presupuesto_cent=None):
        car = self.carrito(cid)
        problemas = []
        if car["estado"] != "abierto":
            problemas.append("el carrito ya fue cerrado por un pedido")
        if not car["items"]:
            problemas.append("el carrito esta vacio")
        for i in car["items"]:
            if i["cantidad"] > i["stock"]:
                problemas.append(f"{i['product_id']}: stock insuficiente (disponibles {i['stock']})")
        if presupuesto_cent is not None and car["total_cent"] > presupuesto_cent:
            problemas.append(f"el total {euros(car['total_cent'])} supera el presupuesto {euros(presupuesto_cent)}")
        cot_id = f"COT-{cid}-{car['version']}"
        items = [{k: i[k] for k in ("product_id", "cantidad", "precio_cent")} for i in car["items"]]
        self.con.execute("INSERT OR REPLACE INTO cotizaciones VALUES (?,?,?,?,?,?,?)",
                         (cot_id, cid, car["version"], car["total_cent"], presupuesto_cent,
                          json.dumps(items), int(not problemas)))
        return {"cotizacion_id": cot_id, "items": car["items"], "total_cent": car["total_cent"],
                "valida": not problemas, "problemas": problemas}

    def pedido(self, pedido_id):
        tot, cot = self.con.execute("SELECT total_cent, cotizacion_id FROM pedidos WHERE id=?", (pedido_id,)).fetchone()
        items = [{"product_id": p, "cantidad": c, "precio_cent": pc} for p, c, pc in self.con.execute(
            "SELECT product_id, cantidad, precio_cent FROM pedido_items WHERE pedido_id=?", (pedido_id,))]
        return {"pedido_id": pedido_id, "cotizacion_id": cot, "total_cent": tot, "items": items}

    def confirmar(self, cot_id, presupuesto_cent=None):
        """Revalida todo dentro de una transaccion. Devuelve estado: registrado | ya_registrado | rechazado."""
        c = self.con
        c.execute("BEGIN IMMEDIATE")
        try:
            prev = c.execute("SELECT id FROM pedidos WHERE cotizacion_id=?", (cot_id,)).fetchone()
            if prev:
                c.execute("ROLLBACK")
                return {"estado": "ya_registrado", **self.pedido(prev[0])}
            cot = c.execute("SELECT carrito_id, version, items FROM cotizaciones WHERE id=?", (cot_id,)).fetchone()
            if cot is None:
                c.execute("ROLLBACK")
                return {"estado": "rechazado", "problemas": ["cotizacion inexistente"]}
            cid, version, items_cot = cot[0], cot[1], {i["product_id"]: i for i in json.loads(cot[2])}
            car = self.carrito(cid)
            problemas = []
            if car["estado"] != "abierto":
                problemas.append("el carrito ya fue cerrado")
            if car["version"] != version:
                problemas.append("el carrito cambio despues de la cotizacion")
            for i in car["items"]:
                if i["cantidad"] > i["stock"]:
                    problemas.append(f"{i['product_id']}: stock insuficiente (pedido {i['cantidad']}, "
                                     f"disponibles {i['stock']})")
                ref = items_cot.get(i["product_id"])
                if ref and ref["precio_cent"] != i["precio_cent"]:
                    problemas.append(f"{i['product_id']}: el precio cambio de {euros(ref['precio_cent'])} "
                                     f"a {euros(i['precio_cent'])}")
            if presupuesto_cent is not None and car["total_cent"] > presupuesto_cent:
                problemas.append(f"el total {euros(car['total_cent'])} supera el presupuesto {euros(presupuesto_cent)}")
            if not car["items"]:
                problemas.append("el carrito esta vacio")
            if problemas:
                c.execute("ROLLBACK")
                return {"estado": "rechazado", "problemas": problemas}
            for i in car["items"]:
                cur = c.execute("UPDATE productos SET stock=stock-? WHERE product_id=? AND stock>=?",
                                (i["cantidad"], i["product_id"], i["cantidad"]))
                if cur.rowcount != 1:
                    raise ErrorTienda("stock modificado durante la confirmacion")
            pid = c.execute("INSERT INTO pedidos(cotizacion_id, carrito_id, total_cent) VALUES (?,?,?)",
                            (cot_id, cid, car["total_cent"])).lastrowid
            c.executemany("INSERT INTO pedido_items VALUES (?,?,?,?)",
                          [(pid, i["product_id"], i["cantidad"], i["precio_cent"]) for i in car["items"]])
            c.execute("UPDATE carritos SET estado='cerrado' WHERE id=?", (cid,))
            c.execute("COMMIT")
            return {"estado": "registrado", **self.pedido(pid)}
        except Exception:
            if c.in_transaction:
                c.execute("ROLLBACK")
            raise

    # --- eventos de escenarios (no los usa el agente) ---
    def fijar_stock(self, pid, stock):
        self.con.execute("UPDATE productos SET stock=? WHERE product_id=?", (int(stock), pid))

    def fijar_precio(self, pid, precio_cent):
        self.con.execute("UPDATE productos SET precio_cent=? WHERE product_id=?", (int(precio_cent), pid))

    def resumen(self):
        """Estado final: pedidos con items, carritos y productos cuyo stock/precio cambio respecto de la base."""
        pedidos = [self.pedido(r[0]) for r in self.con.execute("SELECT id FROM pedidos ORDER BY id")]
        carritos = [self.carrito(r[0]) for r in self.con.execute("SELECT id FROM carritos ORDER BY id")]
        self.con.execute("ATTACH DATABASE ? AS b", (str(self.base),))
        cambios = [{"product_id": p, "stock_base": sb, "stock": s, "precio_base_cent": pb, "precio_cent": pc}
                   for p, sb, s, pb, pc in self.con.execute(
                       "SELECT p.product_id, q.stock, p.stock, q.precio_cent, p.precio_cent FROM productos p "
                       "JOIN b.productos q USING(product_id) WHERE p.stock<>q.stock OR p.precio_cent<>q.precio_cent")]
        self.con.execute("DETACH DATABASE b")
        negativos = self.con.execute("SELECT COUNT(*) FROM productos WHERE stock<0").fetchone()[0]
        return {"huella": self.huella(), "pedidos": pedidos, "carritos": carritos,
                "productos_modificados": cambios, "stock_negativo": negativos}

In [7]:
BASE, TRABAJO = C.RUTA_TIENDA / "tienda_base.sqlite", C.RUTA_TIENDA / "tienda.sqlite"
CFG_TIENDA = {"datos": H_DATOS, "semilla": C.SEMILLA, "precio": C.PRECIO_LOGNORMAL, "rango": C.PRECIO_RANGO,
              "agotado": C.PROB_AGOTADO, "stock": C.STOCK_MAX}
BIT.ejecutar("3. Tienda simulada (base con semilla)", CFG_TIENDA, [BASE],
             lambda: Tienda.crear_base(BASE, CAT["product_id"], C.SEMILLA, *C.PRECIO_LOGNORMAL, C.PRECIO_RANGO,
                                       C.PROB_AGOTADO, C.STOCK_MAX))
TIENDA = Tienda(BASE, TRABAJO, C.CANTIDAD_MAX)
with BIT.paso("3.1 Restauracion reproducible") as info:
    TIENDA.restaurar(); h1 = TIENDA.huella()
    TIENDA.fijar_stock(CAT["product_id"][0], 0)
    TIENDA.restaurar(); h2 = TIENDA.huella()
    assert h1 == h2
    info["comentario"] = f"huella {h1}"
inv = pd.read_sql("SELECT precio_cent / 100.0 AS precio, stock FROM productos", TIENDA.con)
print(f"Huella: {h1} | agotados: {(inv['stock'] == 0).mean():.1%}")
inv.describe().round(2).T

> 3. Tienda simulada (base con semilla)
  OMITIDO | 00:00:00.001
> 3.1 Restauracion reproducible
  OK | 00:00:00.028
Huella: afdd683c93168902 | agotados: 7.9%


,count,mean,std,min,25%,50%,75%,max
precio,10688.0,28.09,25.75,1.99,11.99,20.99,34.99,363.99
stock,10688.0,11.88,7.77,0.00,5.00,12.00,19.00,25.00


## 4. Índice RAG
- **BM25** (modelo base léxico) sobre título, marca, color, características y descripción.
- **Denso:** `MODELO_EMB` (e5, prefijos `query:`/`passage:`); los embeddings del catálogo se guardan en `output/indice/`.
- **Reranker:** `MODELO_RERANK` (cross-encoder multilingüe) sobre los mejores `TOP_RERANK` candidatos híbridos.

In [8]:
# ===== Recuperacion: BM25, denso, hibrido, reranker y nDCG =====
"""Recuperacion sobre el catalogo: BM25 (modelo base lexico), denso (e5), hibrido y reranker.

Metodos (mismos para evaluacion y para el agente):
  bm25            modelo base lexico
  denso           e5 (coseno)
  hibrido         alpha * denso + (1 - alpha) * bm25         (min-max por consulta)
  bm25+reranker   beta * reranker + (1 - beta) * bm25        (ablacion: sin denso)
  sistema         beta * reranker + (1 - beta) * hibrido     (completo)
alpha y beta se ajustan SOLO con consultas dev.
"""
import hashlib
import re
import unicodedata

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

STOP = set("a al ante con contra de del desde el en entre es la las lo los o para por que se sin sobre su sus "
           "un una unas unos y e u".split())
METODOS = ["bm25", "denso", "hibrido", "bm25+reranker", "sistema"]


def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t).lower())
    return "".join(c for c in t if not unicodedata.combining(c))


def tokens(t):
    return [w for w in re.findall(r"[a-z0-9]+", normalizar(t)) if w not in STOP]


def minmax(x):
    x = np.asarray(x, dtype=float)
    r = x.max() - x.min() if len(x) else 0
    return (x - x.min()) / r if r > 0 else np.zeros_like(x)


def ndcg(ganancias_rank, ganancias_todas, k):
    g = np.asarray(ganancias_rank[:k], dtype=float)
    ideal = np.sort(np.asarray(ganancias_todas, dtype=float))[::-1][:k]
    desc = 1 / np.log2(np.arange(2, k + 2))
    idcg = (ideal * desc[:len(ideal)]).sum()
    return float((g * desc[:len(g)]).sum() / idcg) if idcg > 0 else 0.0


class Recuperador:
    def __init__(self, catalogo, max_chars=600):
        self.cat = catalogo.reset_index(drop=True)
        self.ids = self.cat["product_id"].tolist()
        self.pos = {p: i for i, p in enumerate(self.ids)}
        c = self.cat
        self.docs = (c["titulo"] + ". " + c["marca"] + ". " + c["color"] + ". "
                     + c["caracteristicas"].str[:max_chars] + " " + c["descripcion"].str[:max_chars // 2]).tolist()
        self.docs_rr = (c["titulo"] + ". " + c["marca"] + ". " + c["color"] + ". "
                        + c["caracteristicas"].str[:300]).tolist()
        self.bm25 = BM25Okapi([tokens(d) for d in self.docs])
        self.encoder = self.emb = self.reranker = None
        self.alpha, self.beta = 0.5, 0.5

    # --- modelos ---
    def cargar_denso(self, modelo, dispositivo, ruta_cache):
        from sentence_transformers import SentenceTransformer
        self.encoder = SentenceTransformer(modelo, device=dispositivo)
        h = hashlib.sha256((modelo + "|".join(self.ids)).encode()).hexdigest()[:12]
        ruta = ruta_cache / f"emb_{h}.npy"
        if ruta.exists():
            self.emb = np.load(ruta)
        else:
            self.emb = self.encoder.encode(["passage: " + d for d in self.docs], batch_size=64,
                                           normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
            ruta_cache.mkdir(parents=True, exist_ok=True)
            np.save(ruta, self.emb)
        return ruta

    def cargar_reranker(self, modelo, dispositivo):
        from sentence_transformers import CrossEncoder
        self.reranker = CrossEncoder(modelo, device=dispositivo, max_length=256)
        if str(dispositivo).startswith("cuda"):
            self.reranker.model.half()

    # --- puntajes ---
    def s_bm25(self, q, idx=None):
        t = tokens(q)
        return self.bm25.get_scores(t) if idx is None else np.asarray(self.bm25.get_batch_scores(t, list(idx)))

    def s_denso(self, q, idx=None):
        v = self.encoder.encode(["query: " + q], normalize_embeddings=True)[0]
        return self.emb @ v if idx is None else self.emb[list(idx)] @ v

    def s_rerank(self, q, idx):
        return np.asarray(self.reranker.predict([(q, self.docs_rr[i]) for i in idx], batch_size=32,
                                                show_progress_bar=False), dtype=float)

    @staticmethod
    def combinar(metodo, bm, den=None, rr=None, alpha=0.5, beta=0.5):
        if metodo == "bm25":
            return np.asarray(bm, dtype=float)
        if metodo == "denso":
            return np.asarray(den, dtype=float)
        hib = alpha * minmax(den) + (1 - alpha) * minmax(bm) if den is not None else minmax(bm)
        if metodo == "hibrido":
            return hib
        base = minmax(bm) if metodo == "bm25+reranker" else hib
        return beta * minmax(rr) + (1 - beta) * base

    def buscar(self, consulta, k=10, metodo="sistema", top_cand=50, top_rr=30):
        """Top-k sobre todo el catalogo. Devuelve indices de catalogo ordenados."""
        bm = self.s_bm25(consulta)
        if metodo == "bm25":
            return list(np.argsort(-bm)[:k])
        usa_denso = self.encoder is not None and metodo != "bm25+reranker"   # ablacion: candidatos solo BM25
        den = self.s_denso(consulta) if usa_denso else None
        cand = set(np.argsort(-bm)[:top_cand])
        if den is not None:
            cand |= set(np.argsort(-den)[:top_cand])
        if metodo == "denso":
            return list(np.argsort(-den)[:k])
        cand = np.array(sorted(cand))
        s = self.combinar("hibrido", bm[cand], None if den is None else den[cand], alpha=self.alpha)
        orden = cand[np.argsort(-s)]
        if metodo == "hibrido" or self.reranker is None:
            return list(orden[:k])
        sub = orden[:top_rr]
        s = self.combinar(metodo, bm[sub], None if den is None else den[sub], self.s_rerank(consulta, sub),
                          self.alpha, self.beta)
        return list(sub[np.argsort(-s)][:k])


# --- evaluacion ---
def precomputar(rec, consultas, qrels, ganancias):
    """Puntajes de cada metodo sobre los candidatos anotados de cada consulta (se calculan una sola vez)."""
    out = {}
    q = qrels.assign(g=qrels["esci_label"].map(ganancias), i=qrels["product_id"].map(rec.pos))
    for qid, txt in zip(consultas["query_id"], consultas["query"]):
        f = q[q["query_id"] == qid]
        idx = f["i"].to_numpy()
        out[qid] = {"query": txt, "g": f["g"].to_numpy(), "etiquetas": f["esci_label"].to_numpy(),
                    "ids": f["product_id"].to_numpy(), "bm": rec.s_bm25(txt, idx),
                    "den": rec.s_denso(txt, idx) if rec.encoder is not None else None,
                    "rr": rec.s_rerank(txt, idx) if rec.reranker is not None else None}
    return out


def evaluar(pre, metodo, alpha, beta, k):
    filas = []
    for qid, d in pre.items():
        s = Recuperador.combinar(metodo, d["bm"], d["den"], d["rr"], alpha, beta)
        orden = np.argsort(-s, kind="stable")
        filas.append({"query_id": qid, "query": d["query"], "ndcg": ndcg(d["g"][orden], d["g"], k),
                      "top3": " | ".join(f"{d['etiquetas'][i]}" for i in orden[:3])})
    return pd.DataFrame(filas)


def ajustar(pre_dev, k, grid):
    """alpha en hibrido y beta en sistema / bm25+reranker, maximizando nDCG@k en dev."""
    tabla, mejor = [], {}
    if next(iter(pre_dev.values()))["den"] is not None:
        for a in grid:
            tabla.append({"metodo": "hibrido", "alpha": a, "beta": None,
                          "ndcg_dev": evaluar(pre_dev, "hibrido", a, 0, k)["ndcg"].mean()})
    t = pd.DataFrame(tabla)
    mejor["alpha"] = float(t.loc[t["ndcg_dev"].idxmax(), "alpha"]) if len(t) else 0.0
    if next(iter(pre_dev.values()))["rr"] is not None:
        for m in ("sistema", "bm25+reranker"):
            for b in grid:
                tabla.append({"metodo": m, "alpha": mejor["alpha"], "beta": b,
                              "ndcg_dev": evaluar(pre_dev, m, mejor["alpha"], b, k)["ndcg"].mean()})
        t = pd.DataFrame(tabla)
        for m in ("sistema", "bm25+reranker"):
            s = t[t["metodo"] == m]
            mejor[f"beta_{m}"] = float(s.loc[s["ndcg_dev"].idxmax(), "beta"])
    return mejor, pd.DataFrame(tabla)


def evaluar_catalogo(rec, consultas, qrels, metodo, ganancias, k, beta=None):
    """nDCG@k recuperando de todo el catalogo (productos no anotados = 0). Complementario al oficial."""
    if beta is not None:
        rec.beta = beta
    filas = []
    for qid, txt in zip(consultas["query_id"], consultas["query"]):
        f = qrels[qrels["query_id"] == qid]
        gmap = dict(zip(f["product_id"], f["esci_label"].map(ganancias)))
        top = [rec.ids[i] for i in rec.buscar(txt, k, metodo)]
        filas.append({"query_id": qid, "ndcg": ndcg([gmap.get(p, 0.0) for p in top], list(gmap.values()), k)})
    return pd.DataFrame(filas)

In [9]:
CARGA = []   # carga inicial de cada modelo (separada de las consultas posteriores)
def registrar_carga(nombre, etapa, fn):
    v0, t0 = vram(), time.perf_counter()
    obj = fn()
    CARGA.append({"modelo": nombre, "etapa": etapa, "carga_s": round(time.perf_counter() - t0, 2),
                  "vram_gb": round(vram() - v0, 2), "vram_total_gb": vram()})
    print(CARGA[-1])
    return obj

with BIT.paso("4.1 Indice BM25"):
    REC = registrar_carga("BM25", "recuperacion", lambda: Recuperador(CAT, C.MAX_CHARS_TEXTO))
with BIT.paso("4.2 Modelo denso + embeddings del catalogo", C.MODELO_EMB):
    registrar_carga(C.MODELO_EMB, "recuperacion", lambda: REC.cargar_denso(C.MODELO_EMB, DISP, C.RUTA_INDICE))
with BIT.paso("4.3 Reranker", C.MODELO_RERANK):
    registrar_carga(C.MODELO_RERANK, "recuperacion", lambda: REC.cargar_reranker(C.MODELO_RERANK, DISP))
print("Embeddings:", REC.emb.shape)

> 4.1 Indice BM25
{'modelo': 'BM25', 'etapa': 'recuperacion', 'carga_s': 0.62, 'vram_gb': 0.0, 'vram_total_gb': 0.0}
  OK | 00:00:00.625
> 4.2 Modelo denso + embeddings del catalogo
{'modelo': 'intfloat/multilingual-e5-base', 'etapa': 'recuperacion', 'carga_s': 3.55, 'vram_gb': 1.04, 'vram_total_gb': 1.04}
  OK | 00:00:03.549
> 4.3 Reranker
{'modelo': 'BAAI/bge-reranker-v2-m3', 'etapa': 'recuperacion', 'carga_s': 2.93, 'vram_gb': 1.05, 'vram_total_gb': 2.09}
  OK | 00:00:02.930
Embeddings: (10688, 768)


## 5. Evaluación de la recuperación: modelo base y ablación
- **Métrica:** nDCG@10 sobre los **candidatos anotados** de cada consulta, ganancias E = 1, S = 0,1, C = 0,01, I = 0.
- **Ajuste solo con dev:** α (peso denso vs BM25) y β (peso reranker). Las consultas eval se usan una sola vez.
- **Comparación:** BM25 (modelo base) vs denso vs híbrido vs **sistema** (híbrido + reranker).
- **Ablación** (mismas consultas): sistema **sin reranker** (= híbrido) y sistema **sin denso** (BM25 + reranker).
- Complemento: nDCG@10 recuperando de **todo el catálogo** (no anotados = 0), que es lo que vive el agente.

In [10]:
RUTA_REC = C.RUTA_RES / "recuperacion_eval.csv"
CFG_REC = {"datos": H_DATOS, "emb": C.MODELO_EMB, "rr": C.MODELO_RERANK, "grid": C.GRID, "k": C.K_NDCG,
           "gan": C.GANANCIAS, "top": [C.TOP_CANDIDATOS, C.TOP_RERANK]}

def evaluar_recuperacion():
    dev, ev = CONS[CONS["conjunto"] == "dev"], CONS[CONS["conjunto"] == "eval"]
    pre_dev = precomputar(REC, dev, QRELS, C.GANANCIAS)
    mejor, grid = ajustar(pre_dev, C.K_NDCG, C.GRID)
    grid.to_csv(C.RUTA_RES / "recuperacion_ajuste_dev.csv", index=False)
    (C.RUTA_RES / "recuperacion_parametros.json").write_text(json.dumps(mejor, indent=1), encoding="utf-8")
    t0 = time.perf_counter(); pre_ev = precomputar(REC, ev, QRELS, C.GANANCIAS)
    t_pre = (time.perf_counter() - t0) / len(ev)
    variantes = [("BM25 (modelo base)", "bm25", 0, 0), ("Denso e5", "denso", 0, 0),
                 ("Hibrido = sistema sin reranker (ablacion)", "hibrido", mejor["alpha"], 0),
                 ("BM25 + reranker = sistema sin denso (ablacion)", "bm25+reranker", 0, mejor["beta_bm25+reranker"]),
                 ("Sistema: hibrido + reranker", "sistema", mejor["alpha"], mejor["beta_sistema"])]
    filas, por_q = [], []
    for nombre, m, a, b in variantes:
        r_dev = evaluar(pre_dev, m, a, b, C.K_NDCG)["ndcg"].mean()
        r = evaluar(pre_ev, m, a, b, C.K_NDCG)
        por_q.append(r.assign(metodo=m))
        REC.alpha = a
        cat_ev = evaluar_catalogo(REC, ev, QRELS, m, C.GANANCIAS, C.K_NDCG, beta=b)["ndcg"].mean() if m != "denso" or REC.encoder else np.nan
        filas.append({"variante": nombre, "metodo": m, "alpha": a, "beta": b, "nDCG@10 dev": r_dev,
                      "nDCG@10 eval (candidatos anotados)": r["ndcg"].mean(), "nDCG@10 eval (catalogo completo)": cat_ev})
    pd.DataFrame(filas).to_csv(RUTA_REC, index=False)
    pd.concat(por_q).to_csv(C.RUTA_RES / "recuperacion_por_consulta.csv", index=False)

if C.EJECUTAR_EVAL_RECUPERACION:
    BIT.ejecutar("5. Evaluacion de recuperacion (ajuste dev, eval una vez)", CFG_REC, [RUTA_REC], evaluar_recuperacion)
PARAM = json.loads((C.RUTA_RES / "recuperacion_parametros.json").read_text())
REC.alpha, REC.beta = PARAM["alpha"], PARAM.get("beta_sistema", 0.5)    # configuracion que usa el agente
RES_REC = pd.read_csv(RUTA_REC)
base = RES_REC.loc[RES_REC["metodo"] == "bm25", "nDCG@10 eval (candidatos anotados)"].iloc[0]
RES_REC["mejora vs BM25"] = RES_REC["nDCG@10 eval (candidatos anotados)"] - base
print("Parametros elegidos en dev:", PARAM)
RES_REC.round(4)

> 5. Evaluacion de recuperacion (ajuste dev, eval una vez)
  OMITIDO | 00:00:00.002
Parametros elegidos en dev: {'alpha': 0.7, 'beta_sistema': 0.8, 'beta_bm25+reranker': 1.0}


,variante,metodo,alpha,beta,nDCG@10 dev,nDCG@10 eval (candidatos anotados),nDCG@10 eval (catalogo completo),mejora vs BM25
0,BM25 (modelo base),bm25,0.0,0.0,0.6924,0.6925,0.5623,0.0000
1,Denso e5,denso,0.0,0.0,0.7257,0.7273,0.5971,0.0348
2,Hibrido = sistema sin reranker (ablacion),hibrido,0.7,0.0,0.7380,0.7323,0.6335,0.0398
3,BM25 + reranker = sistema sin denso (ablacion),bm25+reranker,0.0,1.0,0.7564,0.7687,0.6566,0.0761
4,Sistema: hibrido + reranker,sistema,0.7,0.8,0.7592,0.7656,0.6851,0.0731


## 6. Modelos de lenguaje y voz (carga inicial y memoria)

| Etapa | Modelo | Por qué | VRAM aprox. |
|---|---|---|---|
| ASR | `openai/whisper-large-v3-turbo` (fp16) | Mejor WER en español que small/medium con 4 capas de decodificador: rápido | ~1,6 GB |
| LLM / agente | `Qwen/Qwen2.5-7B-Instruct` (nf4) | Llamado a herramientas nativo (`<tool_call>`), buen español, cabe en 16 GB cuantizado | ~5,5 GB |
| Embeddings | `intfloat/multilingual-e5-base` | Multilingüe, buen equilibrio calidad/latencia | ~1,1 GB |
| Reranker | `BAAI/bge-reranker-v2-m3` (fp16) | Cross-encoder multilingüe de alto rendimiento | ~1,1 GB |
| TTS | XTTS-v2 (coqui-tts) | Clonación de voz en español con 6–10 s de referencia, sin entrenar | ~2 GB |
| Semejanza | `microsoft/wavlm-base-plus-sv` | x-vectors para verificar el parecido con la voz de referencia | CPU |

Gestión de memoria: el LLM va en 4 bits; los latentes de la voz de referencia se calculan una sola vez; los tiempos y la
VRAM de cada carga quedan en la tabla `carga_modelos.csv`, separados de las consultas.

In [11]:
# ===== LLM local con llamado a herramientas =====
"""Backends del LLM. LLMLocal: transformers + chat template con herramientas (formato Qwen/Hermes <tool_call>).
LLMGuion: respuestas predefinidas, solo para la autoprueba sin modelos."""
import json
import re

TOOL_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)


def _llamada(obj):
    if not isinstance(obj, dict) or "name" not in obj:
        return None
    args = obj.get("arguments", obj.get("parameters", {}))
    if isinstance(args, str):
        try:
            args = json.loads(args) if args.strip() else {}
        except json.JSONDecodeError:
            args = {"_crudo": args}
    return {"nombre": obj["name"], "argumentos": args if isinstance(args, dict) else {}}


def analizar_salida(texto):
    """Extrae llamadas <tool_call>{json}</tool_call>; si no hay, intenta un JSON suelto con 'name'."""
    llamadas = []
    for bloque in TOOL_RE.findall(texto):
        try:
            c = _llamada(json.loads(bloque))
        except json.JSONDecodeError:
            c = None
        llamadas.append(c or {"nombre": "_invalida", "argumentos": {"_crudo": bloque[:200]}})
    limpio = TOOL_RE.sub("", texto).replace("<tool_call>", "").strip()
    if not llamadas and limpio.startswith("{"):
        try:
            c = _llamada(json.loads(limpio))
            if c:
                llamadas, limpio = [c], ""
        except json.JSONDecodeError:
            pass
    return {"texto": limpio, "llamadas": llamadas, "crudo": texto}


class LLMLocal:
    def __init__(self, modelo, cuatro_bits=True, max_tokens=384):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(modelo)
        usar_cuda = str(DISP).startswith("cuda")
        kw = {"device_map": "auto"} if usar_cuda else {}
        if cuatro_bits and usar_cuda:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16,
                                                           bnb_4bit_use_double_quant=True)
        else:
            kw["torch_dtype"] = torch.float16 if usar_cuda else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(modelo, **kw).eval()
        self.max_tokens = max_tokens

    def generar(self, mensajes, herramientas):
        enc = self.tok.apply_chat_template(mensajes, tools=herramientas, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt")
        enc = {k: enc[k].to(self.model.device) for k in ("input_ids", "attention_mask") if k in enc}
        with self.torch.inference_mode():
            out = self.model.generate(**enc, max_new_tokens=self.max_tokens, do_sample=False,
                                      temperature=None, top_p=None, top_k=None,
                                      pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id)
        texto = self.tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=False)
        for fin in ("<|im_end|>", "<|endoftext|>", "<|eot_id|>", self.tok.eos_token or ""):
            texto = texto.replace(fin, "") if fin else texto
        return analizar_salida(texto.strip())


class LLMGuion:
    """Devuelve salidas predefinidas en orden (texto crudo con <tool_call>)."""
    def __init__(self, salidas):
        self.salidas = list(salidas)

    def generar(self, mensajes, herramientas):
        s = self.salidas.pop(0)
        return analizar_salida(s(mensajes) if callable(s) else s)

In [12]:
# ===== Voz: ASR, TTS clonado, semejanza y metricas =====
"""Voz: ASR (Whisper), TTS con clonacion (XTTS-v2), semejanza de voz (WavLM x-vector) y metricas.

Metricas:
- wer / cer normalizados (minusculas, sin tildes ni puntuacion).
- errores en entidades: cantidades, importes (numeros seguidos de euros) y terminos de producto.
- inteligibilidad del TTS: WER entre el texto sintetizado y su re-transcripcion con el ASR.
- semejanza: coseno entre x-vectors de la voz sintetizada y la referencia.
"""
import os
import re
from collections import Counter

import numpy as np



def cargar_audio(ruta, sr=16000):
    import soundfile as sf
    x, s = sf.read(str(ruta), dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if s != sr:
        import librosa
        x = librosa.resample(x, orig_sr=s, target_sr=sr)
    return x


def grabar(ruta, segundos=7, sr=16000):
    """Graba del microfono (sounddevice) y guarda WAV. La medicion de latencia empieza al terminar."""
    import sounddevice as sd
    import soundfile as sf
    x = sd.rec(int(segundos * sr), samplerate=sr, channels=1, dtype="float32")
    sd.wait()
    sf.write(str(ruta), x[:, 0], sr)
    return ruta


class ASR:
    def __init__(self, modelo, dispositivo):
        import torch
        from transformers import pipeline
        dtype = torch.float16 if str(dispositivo).startswith("cuda") else torch.float32
        self.p = pipeline("automatic-speech-recognition", model=modelo, torch_dtype=dtype, device=dispositivo)

    def transcribir(self, ruta):
        audio = cargar_audio(ruta, 16000)
        out = self.p({"raw": audio, "sampling_rate": 16000}, chunk_length_s=30,
                     generate_kwargs={"language": "spanish", "task": "transcribe"})
        return out["text"].strip()


class TTSClonado:
    def __init__(self, modelo, ruta_ref, texto_ref, dispositivo):
        os.environ.setdefault("COQUI_TOS_AGREED", "1")   # licencia CPML de XTTS (uso no comercial)
        from TTS.api import TTS
        self.api = TTS(modelo).to(dispositivo)
        self.m = self.api.synthesizer.tts_model
        self.sr = self.api.synthesizer.output_sample_rate
        self.lat, self.spk = self.m.get_conditioning_latents(audio_path=[str(ruta_ref)])  # una sola vez
        self.ref = set(palabras(texto_ref))

    def sintetizar(self, texto, ruta, semilla=0):
        import soundfile as sf
        import torch
        p = set(palabras(texto))
        if not p or len(p & self.ref) / len(p | self.ref) > 0.8:
            raise ValueError("el texto a sintetizar coincide con el de la referencia")
        torch.manual_seed(semilla)
        wav = self.m.inference(texto, "es", self.lat, self.spk, temperature=0.7, enable_text_splitting=True)["wav"]
        sf.write(str(ruta), np.asarray(wav, dtype=np.float32), self.sr)
        return ruta


class Semejanza:
    def __init__(self, modelo, dispositivo="cpu"):
        from transformers import AutoFeatureExtractor, WavLMForXVector
        self.fe = AutoFeatureExtractor.from_pretrained(modelo)
        self.m = WavLMForXVector.from_pretrained(modelo).to(dispositivo).eval()
        self.disp = dispositivo

    def embedding(self, ruta):
        import torch
        x = cargar_audio(ruta, 16000)
        enc = self.fe(x, sampling_rate=16000, return_tensors="pt").to(self.disp)
        with torch.inference_mode():
            e = self.m(**enc).embeddings[0]
        return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

    def coseno(self, a, b):
        return float(np.dot(self.embedding(a), self.embedding(b)))


# --- metricas de texto ---
def palabras(t):
    return re.findall(r"[a-z0-9]+(?:[.,][0-9]+)?|€", normalizar(t))


def wer(ref, hip):
    import jiwer
    r, h = " ".join(palabras(ref)), " ".join(palabras(hip))
    return float(jiwer.wer(r, h)) if r else float(bool(h))


def cer(ref, hip):
    import jiwer
    r, h = " ".join(palabras(ref)), " ".join(palabras(hip))
    return float(jiwer.cer(r, h)) if r else float(bool(h))


UNI = {"cero": 0, "un": 1, "uno": 1, "una": 1, "dos": 2, "tres": 3, "cuatro": 4, "cinco": 5, "seis": 6, "siete": 7,
       "ocho": 8, "nueve": 9, "diez": 10, "once": 11, "doce": 12, "trece": 13, "catorce": 14, "quince": 15,
       "dieciseis": 16, "diecisiete": 17, "dieciocho": 18, "diecinueve": 19, "veinte": 20, "veintiun": 21,
       "veintiuno": 21, "veintiuna": 21, "veintidos": 22, "veintitres": 23, "veinticuatro": 24, "veinticinco": 25,
       "veintiseis": 26, "veintisiete": 27, "veintiocho": 28, "veintinueve": 29}
DEC = {"treinta": 30, "cuarenta": 40, "cincuenta": 50, "sesenta": 60, "setenta": 70, "ochenta": 80, "noventa": 90}
CEN = {"cien": 100, "ciento": 100, "doscientos": 200, "doscientas": 200, "trescientos": 300, "trescientas": 300,
       "cuatrocientos": 400, "cuatrocientas": 400, "quinientos": 500, "quinientas": 500, "seiscientos": 600,
       "seiscientas": 600, "setecientos": 700, "setecientas": 700, "ochocientos": 800, "ochocientas": 800,
       "novecientos": 900, "novecientas": 900}
MONEDA = {"euro", "euros", "€", "eur"}


def _es_num(w):
    return w in UNI or w in DEC or w in CEN or w == "mil" or re.fullmatch(r"\d+(?:[.,]\d+)?", w) is not None


def _leer_numero(ws, i):
    """Lee un numero (palabras o digitos) desde ws[i]. Devuelve (valor, siguiente_indice)."""
    w = ws[i]
    if re.fullmatch(r"\d+(?:[.,]\d+)?", w):
        return float(w.replace(",", ".")), i + 1
    total, actual = 0, 0
    while i < len(ws):
        w = ws[i]
        if w in CEN:
            actual += CEN[w]
        elif w in DEC:
            actual += DEC[w]
        elif w in UNI:
            actual += UNI[w]
        elif w == "mil":
            total += max(actual, 1) * 1000
            actual = 0
        elif w == "y" and i + 1 < len(ws) and ws[i + 1] in UNI and actual % 10 == 0 and actual >= 30:
            pass
        else:
            break
        i += 1
    return float(total + actual), i


def entidades(texto):
    """Cantidades e importes (Counter de valores) y terminos de producto (set) de una frase."""
    ws = palabras(texto)
    cant, imp, i = Counter(), Counter(), 0
    while i < len(ws):
        if not _es_num(ws[i]):
            i += 1
            continue
        v, j = _leer_numero(ws, i)
        if j < len(ws) - 1 and ws[j] == "con" and _es_num(ws[j + 1]):        # "doce con cincuenta"
            d, j = _leer_numero(ws, j + 1)
            v += d / 100 if d < 100 else 0
        es_imp = (j < len(ws) and ws[j] in MONEDA) or (i > 0 and ws[i - 1] == "€")
        (imp if es_imp else cant)[round(v, 2)] += 1
        i = j
    conversa = {"quiero", "comprar", "compra", "agrega", "agregar", "anade", "anadir", "pon", "carrito", "confirmo",
                "confirma", "confirmar", "si", "vale", "presupuesto", "maximo", "total", "cotizacion", "pedido",
                "favor", "opcion", "primera", "segunda", "tercera", "mejor", "entonces", "dame", "prepara",
                "preparame", "necesito", "busco", "unidades", "unidad", "mas", "menos", "barato", "barata",
                "euros", "euro", "con", "hola", "gracias", "otra", "otro", "vez", "esa", "ese", "eso", "por",
                "son", "sean", "dije", "perdon", "cambia", "cambiala", "quita", "tambien", "solo", "cuesta"}
    prod = {w for w in ws if len(w) >= 3 and w not in STOP and w not in conversa and not _es_num(w)}
    return {"cantidades": cant, "importes": imp, "productos": prod}


def errores_entidades(ref, hip):
    a, b = entidades(ref), entidades(hip)
    dif = lambda x, y: max(sum((x - y).values()), sum((y - x).values()))   # sustitucion = 1 error
    return {"err_cantidades": dif(a["cantidades"], b["cantidades"]), "err_importes": dif(a["importes"], b["importes"]),
            "err_productos": len(a["productos"] - b["productos"]),
            "n_cantidades": sum(a["cantidades"].values()), "n_importes": sum(a["importes"].values()),
            "n_productos": len(a["productos"])}

In [13]:
MOD = {}
def obtener(nombre):
    """Carga perezosa: cada modelo se carga una vez y se registra su tiempo de carga inicial."""
    if nombre not in MOD:
        if nombre == "llm":
            MOD[nombre] = registrar_carga(C.MODELO_LLM, "agente", lambda: LLMLocal(C.MODELO_LLM, C.LLM_4BIT, C.LLM_MAX_TOKENS))
        elif nombre == "asr":
            MOD[nombre] = registrar_carga(C.MODELO_ASR, "asr", lambda: ASR(C.MODELO_ASR, DISP))
        elif nombre == "tts":
            assert C.RUTA_VOZ_REF.exists(), f"Falta la grabacion de referencia: {C.RUTA_VOZ_REF}"
            MOD[nombre] = registrar_carga(C.MODELO_TTS, "tts", lambda: TTSClonado(C.MODELO_TTS, C.RUTA_VOZ_REF, C.TEXTO_REF, DISP))
        elif nombre == "semejanza":
            MOD[nombre] = registrar_carga(C.MODELO_SEMEJANZA, "metricas", lambda: Semejanza(C.MODELO_SEMEJANZA, "cpu"))
    return MOD[nombre]

with BIT.paso("6. Carga de LLM"):
    obtener("llm")
pd.DataFrame(CARGA)

> 6. Carga de LLM


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

{'modelo': 'Qwen/Qwen2.5-7B-Instruct', 'etapa': 'agente', 'carga_s': 19.8, 'vram_gb': 5.19, 'vram_total_gb': 7.28}
  OK | 00:00:19.192


,modelo,etapa,carga_s,vram_gb,vram_total_gb
0,BM25,recuperacion,0.62,0.00,0.00
1,intfloat/multilingual-e5-base,recuperacion,3.55,1.04,1.04
2,BAAI/bge-reranker-v2-m3,recuperacion,2.93,1.05,2.09
3,Qwen/Qwen2.5-7B-Instruct,agente,19.80,5.19,7.28


## 7. Agente, herramientas, memoria y validaciones
- **Herramientas:** `buscar_productos` (RAG), `ver_producto`, `consultar_atributo` (atributo no verificable = desconocido),
  `registrar_restricciones`, `agregar_al_carrito`, `cambiar_cantidad`, `ver_carrito`, `cotizar`, `confirmar_pedido`.
- **Memoria:** presupuesto, preferencias, carrito, cotización vigente, pedidos y productos mostrados; se inyecta en cada turno.
- **Validaciones por código:** enteros 1..`CANTIDAD_MAX`, stock, presupuesto, consentimiento explícito sobre una cotización ya
  mostrada, confirmación atómica (sin cambios parciales) e idempotente (sin pedidos duplicados).
- La autoprueba (celda siguiente) ejecuta todo esto sin modelos (BM25 + LLM con guion).

In [14]:
# ===== Herramientas, memoria (Estado) y validaciones por codigo =====
"""Herramientas del agente, memoria (Estado) y validaciones por codigo.

Cada herramienta recibe argumentos del LLM, los valida y devuelve un dict JSON con ok / error.
Las etiquetas ESCI no estan aqui: el agente solo ve catalogo + tienda.
"""
import re
import time
from dataclasses import dataclass, field


NEGACION = re.compile(r"^\s*no\b|\bno (lo |la )?(confirm|compr|quiero)|\bmejor no\b|\bcancel|\bespera\b|\btodavia no\b")
AFIRMACION = re.compile(r"\b(si|confirmo|confirma|confirmar|confirmado|confirmalo|adelante|procede|dale|de acuerdo|"
                        r"ok|okay|vale|compralo|comprala|hazlo|perfecto|correcto)\b")


def es_confirmacion(texto):
    """Consentimiento explicito del usuario (validacion por codigo, no por el LLM)."""
    t = normalizar(texto)
    return bool(AFIRMACION.search(t)) and not NEGACION.search(t)


@dataclass
class Estado:
    presupuesto_cent: int = None
    preferencias: list = field(default_factory=list)
    carrito_id: int = None
    cotizacion: dict = None            # ultima cotizacion {cotizacion_id, total_cent, valida}
    cotizacion_mostrada: bool = False  # se presento al usuario en un turno anterior
    cotizo_en_turno: bool = False
    pedidos: list = field(default_factory=list)
    mostrados: list = field(default_factory=list)   # ultimos productos presentados
    texto_usuario: str = ""
    historial: list = field(default_factory=list)   # [(usuario, asistente)]


def _f(x):
    return None if x is None else euros(x)


ESQUEMAS = [
    {"type": "function", "function": {
        "name": "buscar_productos",
        "description": "Busca productos en el catalogo (RAG). Devuelve titulo, marca, color, precio y existencias.",
        "parameters": {"type": "object", "properties": {
            "consulta": {"type": "string", "description": "Descripcion del producto en palabras clave"},
            "precio_max": {"type": "number", "description": "Precio unitario maximo en euros (opcional)"},
            "k": {"type": "integer", "description": "Cantidad de resultados (1 a 10)"}},
            "required": ["consulta"]}}},
    {"type": "function", "function": {
        "name": "ver_producto",
        "description": "Detalle de un producto: descripcion, caracteristicas, marca, color, precio y existencias.",
        "parameters": {"type": "object", "properties": {"product_id": {"type": "string"}},
                       "required": ["product_id"]}}},
    {"type": "function", "function": {
        "name": "consultar_atributo",
        "description": "Verifica si un atributo (p. ej. 'resistente al agua', '500 ml') aparece en la informacion "
                       "del producto. Si no aparece, el atributo es desconocido.",
        "parameters": {"type": "object", "properties": {"product_id": {"type": "string"},
                                                        "atributo": {"type": "string"}},
                       "required": ["product_id", "atributo"]}}},
    {"type": "function", "function": {
        "name": "registrar_restricciones",
        "description": "Guarda en memoria el presupuesto total maximo y/o las preferencias del usuario. "
                       "preferencias reemplaza la lista completa.",
        "parameters": {"type": "object", "properties": {
            "presupuesto_max": {"type": "number", "description": "Presupuesto total en euros"},
            "preferencias": {"type": "array", "items": {"type": "string"}}}}}},
    {"type": "function", "function": {
        "name": "agregar_al_carrito",
        "description": "Agrega unidades de un producto al carrito (suma a las que ya hubiera).",
        "parameters": {"type": "object", "properties": {"product_id": {"type": "string"},
                                                        "cantidad": {"type": "integer"}},
                       "required": ["product_id", "cantidad"]}}},
    {"type": "function", "function": {
        "name": "cambiar_cantidad",
        "description": "Fija la cantidad de un producto del carrito. 0 lo elimina.",
        "parameters": {"type": "object", "properties": {"product_id": {"type": "string"},
                                                        "cantidad": {"type": "integer"}},
                       "required": ["product_id", "cantidad"]}}},
    {"type": "function", "function": {
        "name": "ver_carrito", "description": "Contenido del carrito y total.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "cotizar",
        "description": "Calcula la cotizacion del carrito y valida existencias y presupuesto. "
                       "Presenta el total y pide confirmacion al usuario.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "confirmar_pedido",
        "description": "Registra el pedido de la ultima cotizacion. Solo si el usuario confirmo explicitamente "
                       "la cotizacion en su ultimo mensaje.",
        "parameters": {"type": "object", "properties": {}}}},
]
MODIFICAN_CARRITO = {"agregar_al_carrito", "cambiar_cantidad", "confirmar_pedido"}


class Herramientas:
    def __init__(self, catalogo, recuperador, tienda, k_resultados=5, metodo="sistema"):
        self.cat = catalogo.set_index("product_id")
        self.rec, self.tienda, self.k, self.metodo = recuperador, tienda, k_resultados, metodo
        self.estado = Estado()
        self.t_recuperacion = 0.0

    # --- utilidades ---
    def _existe(self, pid):
        if not isinstance(pid, str) or pid not in self.cat.index:
            raise ErrorTienda(f"product_id desconocido: {pid!r}. Usa un id devuelto por buscar_productos")

    @staticmethod
    def _numero(x, nombre):
        if isinstance(x, str):
            try:
                x = float(x.replace(",", ".").replace("€", "").strip())
            except ValueError:
                pass
        if isinstance(x, bool) or not isinstance(x, (int, float)) or x <= 0:
            raise ErrorTienda(f"{nombre} debe ser un numero positivo")
        return float(x)

    @staticmethod
    def _entero(x, nombre="cantidad"):
        if isinstance(x, str) and x.strip().lstrip("-").isdigit():
            x = int(x)
        if isinstance(x, float) and x.is_integer():
            x = int(x)
        if x is None:
            raise ErrorTienda(f"falta {nombre}: pregunta al usuario cuantas unidades quiere")
        if not isinstance(x, int) or isinstance(x, bool):
            raise ErrorTienda(f"{nombre} debe ser un numero entero")
        return x

    def _ficha(self, pid, extra=None):
        r = self.cat.loc[pid]
        ps = self.tienda.producto(pid)
        d = {"product_id": pid, "titulo": r["titulo"][:140], "marca": r["marca"] or "desconocido",
             "color": r["color"] or "desconocido", "precio": euros(ps["precio_cent"]), "stock": ps["stock"],
             "disponible": ps["stock"] > 0}
        return {**d, **(extra or {})}

    def _carrito(self):
        e = self.estado
        if e.carrito_id is None:
            return {"items": [], "total": 0.0}
        car = self.tienda.carrito(e.carrito_id)
        return {"items": [{"product_id": i["product_id"], "titulo": self.cat.loc[i["product_id"], "titulo"][:80],
                           "cantidad": i["cantidad"], "precio": euros(i["precio_cent"]), "stock": i["stock"],
                           "subtotal": euros(i["subtotal_cent"])} for i in car["items"]],
                "total": euros(car["total_cent"])}

    def _carrito_abierto(self):
        e = self.estado
        if e.carrito_id is None or self.tienda.carrito(e.carrito_id)["estado"] != "abierto":
            e.carrito_id = self.tienda.nuevo_carrito()
        return e.carrito_id

    def _invalidar_cotizacion(self):
        self.estado.cotizacion_mostrada = self.estado.cotizo_en_turno = False

    # --- herramientas ---
    def buscar_productos(self, consulta, precio_max=None, k=None):
        if not isinstance(consulta, str) or not consulta.strip():
            raise ErrorTienda("consulta vacia")
        k = min(max(self._entero(k if k is not None else self.k, "k"), 1), 10)
        if precio_max is not None:
            precio_max = self._numero(precio_max, "precio_max")
        t0 = time.perf_counter()
        idx = self.rec.buscar(consulta, k=30, metodo=self.metodo)
        self.t_recuperacion += time.perf_counter() - t0
        res = []
        for i in idx:
            f = self._ficha(self.rec.ids[i])
            if precio_max is None or f["precio"] <= precio_max:
                res.append({"posicion": len(res) + 1, **f})
            if len(res) == k:
                break
        self.estado.mostrados = res
        return {"ok": True, "resultados": res,
                "nota": "sin resultados con esos filtros" if not res else "precios unitarios en euros"}

    def ver_producto(self, product_id):
        self._existe(product_id)
        r = self.cat.loc[product_id]
        return {"ok": True, **self._ficha(product_id, {
            "descripcion": r["descripcion"][:500] or "desconocido",
            "caracteristicas": r["caracteristicas"][:500] or "desconocido"})}

    def consultar_atributo(self, product_id, atributo):
        self._existe(product_id)
        r = self.cat.loc[product_id]
        texto = " ".join([r["titulo"], r["marca"], r["color"], r["caracteristicas"], r["descripcion"]])
        objetivo = tokens(atributo)
        if not objetivo:
            raise ErrorTienda("atributo vacio")
        for frase in re.split(r"(?<=[.;\n])\s+", texto):
            if all(t in tokens(frase) for t in objetivo):
                return {"ok": True, "atributo": atributo, "valor": "verificado", "evidencia": frase[:200]}
        return {"ok": True, "atributo": atributo, "valor": "desconocido",
                "nota": "no aparece en la informacion del producto; no afirmes ni niegues el atributo"}

    def registrar_restricciones(self, presupuesto_max=None, preferencias=None):
        e = self.estado
        if presupuesto_max is not None:
            e.presupuesto_cent = int(round(self._numero(presupuesto_max, "presupuesto_max") * 100))
        if preferencias is not None:
            if not isinstance(preferencias, list) or not all(isinstance(p, str) for p in preferencias):
                raise ErrorTienda("preferencias debe ser una lista de textos")
            e.preferencias = [p.strip() for p in preferencias if p.strip()]
        return {"ok": True, "presupuesto_max": _f(e.presupuesto_cent), "preferencias": e.preferencias}

    def _fijar(self, product_id, cantidad):
        cid = self._carrito_abierto()
        self.tienda.fijar_cantidad(cid, product_id, cantidad)
        self._invalidar_cotizacion()
        car = self._carrito()
        out = {"ok": True, "carrito": car}
        p = self.estado.presupuesto_cent
        if p is not None and car["total"] * 100 > p:
            out["aviso"] = f"el total {car['total']} supera el presupuesto {_f(p)}"
        return out

    def agregar_al_carrito(self, product_id, cantidad=None):
        self._existe(product_id)
        cantidad = self._entero(cantidad)
        if cantidad < 1:
            raise ErrorTienda("la cantidad a agregar debe ser al menos 1")
        actual = {i["product_id"]: i["cantidad"] for i in self._carrito()["items"]}.get(product_id, 0)
        return self._fijar(product_id, actual + cantidad)

    def cambiar_cantidad(self, product_id, cantidad=None):
        self._existe(product_id)
        cantidad = self._entero(cantidad)
        if product_id not in {i["product_id"] for i in self._carrito()["items"]}:
            raise ErrorTienda("ese producto no esta en el carrito")
        return self._fijar(product_id, cantidad)

    def ver_carrito(self):
        return {"ok": True, "carrito": self._carrito(), "presupuesto_max": _f(self.estado.presupuesto_cent)}

    def cotizar(self):
        e = self.estado
        if e.carrito_id is None:
            raise ErrorTienda("el carrito esta vacio")
        c = self.tienda.cotizar(e.carrito_id, e.presupuesto_cent)
        e.cotizacion = {"cotizacion_id": c["cotizacion_id"], "total_cent": c["total_cent"], "valida": c["valida"]}
        e.cotizo_en_turno = c["valida"]
        return {"ok": True, "cotizacion_id": c["cotizacion_id"], "carrito": self._carrito(),
                "total": euros(c["total_cent"]), "presupuesto_max": _f(e.presupuesto_cent),
                "valida": c["valida"], "problemas": c["problemas"],
                "siguiente_paso": "presenta el total y pide confirmacion" if c["valida"]
                else "explica los problemas y propone una solucion"}

    def confirmar_pedido(self):
        e = self.estado
        if e.cotizacion is None:
            raise ErrorTienda("no hay cotizacion: arma el carrito y usa cotizar primero")
        ya = any(p["cotizacion_id"] == e.cotizacion["cotizacion_id"] for p in e.pedidos)
        if not ya and not (e.cotizacion_mostrada and es_confirmacion(e.texto_usuario)):
            raise ErrorTienda("el usuario no confirmo explicitamente la cotizacion vigente: presentala y pide "
                              "confirmacion antes de registrar el pedido")
        r = self.tienda.confirmar(e.cotizacion["cotizacion_id"], e.presupuesto_cent)
        if r["estado"] == "rechazado":
            self._invalidar_cotizacion()
            return {"ok": False, "estado": "rechazado", "problemas": r["problemas"], "carrito": self._carrito(),
                    "siguiente_paso": "explica el problema y propone una alternativa; vuelve a cotizar"}
        if r["estado"] == "registrado":
            e.pedidos.append({"pedido_id": r["pedido_id"], "cotizacion_id": r["cotizacion_id"],
                              "total_cent": r["total_cent"]})
            e.carrito_id = None
        return {"ok": True, "estado": r["estado"], "pedido_id": r["pedido_id"], "total": euros(r["total_cent"]),
                "nota": "pedido ya registrado antes; no se duplico" if r["estado"] == "ya_registrado" else
                "pedido registrado e inventario actualizado"}

    # --- despacho ---
    def ejecutar(self, nombre, argumentos):
        fn = getattr(self, nombre, None) if nombre in {s["function"]["name"] for s in ESQUEMAS} else None
        if fn is None:
            return {"ok": False, "error": f"herramienta desconocida: {nombre}"}
        if not isinstance(argumentos, dict):
            return {"ok": False, "error": "argumentos deben ser un objeto JSON"}
        try:
            return fn(**argumentos)
        except TypeError as ex:
            return {"ok": False, "error": f"argumentos invalidos: {ex}"}
        except ErrorTienda as ex:
            return {"ok": False, "error": str(ex)}

    def resumen_estado(self):
        """Memoria que se inyecta en cada turno (sobrevive aunque el historial se recorte)."""
        e = self.estado
        lin = ["ESTADO ACTUAL (memoria del sistema):",
               f"- Presupuesto total maximo: {_f(e.presupuesto_cent) or 'no indicado'} euros",
               f"- Preferencias: {', '.join(e.preferencias) or 'ninguna'}"]
        car = self._carrito()
        if car["items"]:
            lin.append(f"- Carrito (total {car['total']} euros):")
            lin += [f"  * {i['product_id']} | {i['titulo']} | {i['cantidad']} x {i['precio']} | stock {i['stock']}"
                    for i in car["items"]]
        else:
            lin.append("- Carrito: vacio")
        if e.cotizacion:
            lin.append(f"- Ultima cotizacion: {e.cotizacion['cotizacion_id']} total {_f(e.cotizacion['total_cent'])} "
                       f"(valida: {'si' if e.cotizacion['valida'] else 'no'}; mostrada al usuario: "
                       f"{'si' if e.cotizacion_mostrada else 'no'})")
        if e.pedidos:
            lin.append("- Pedidos confirmados: " + ", ".join(f"{p['pedido_id']} ({p['cotizacion_id']}, "
                                                            f"{_f(p['total_cent'])})" for p in e.pedidos))
        if e.mostrados:
            lin.append("- Productos mostrados en la ultima busqueda:")
            lin += [f"  {m['posicion']}) {m['product_id']} | {m['titulo'][:90]} | {m['precio']} euros | "
                    f"stock {m['stock']}" for m in e.mostrados]
        return "\n".join(lin)

In [15]:
# ===== Agente: bucle LLM + herramientas =====
"""Agente de compras: un solo LLM que elige herramientas en un bucle por turno.

Control implementado por el grupo:
- Memoria: Estado (presupuesto, preferencias, carrito, cotizacion, pedidos, productos mostrados) inyectado en
  cada turno + historial recortado.
- Bucle: el LLM propone llamadas a herramientas; el codigo las valida y ejecuta; termina cuando responde texto.
- Reglas duras por codigo (no dependen del LLM): consentimiento explicito, validacion de cantidades,
  presupuesto y existencias, confirmacion atomica e idempotente.
"""
import json
import re
import time


SISTEMA = """Eres un asistente de compras por voz de una tienda en linea. Hablas en espanol y respondes breve \
(maximo 3 frases), porque tu respuesta se leera en voz alta.
Reglas:
1. Usa SIEMPRE las herramientas para obtener productos, precios, existencias y atributos. No inventes datos.
2. Si un atributo no figura en la informacion del producto, dilo como desconocido (usa consultar_atributo).
3. Si falta informacion para actuar (que producto, para que dispositivo, talla, cantidad) o hay una ambiguedad \
relevante, haz UNA pregunta de aclaracion antes de tocar el carrito.
4. Cuando el usuario mencione presupuesto o preferencias, guardalos con registrar_restricciones y respetalos al \
buscar y al elegir.
5. Para comprar: agrega al carrito, usa cotizar, presenta el total y pide confirmacion explicita. Llama a \
confirmar_pedido solo si el ultimo mensaje del usuario confirma la cotizacion ya presentada.
6. Si un producto esta agotado o no alcanza el stock, propone una alternativa disponible o explica que no hay solucion.
7. Si confirmar_pedido informa problemas (stock o precio cambiados), explicalo, propone una solucion y vuelve a \
cotizar. Si el pedido ya existe, no lo repitas.
8. Si el usuario cambia una cantidad o preferencia, conserva las demas restricciones.
9. Nombra los productos por su nombre corto y precio; no leas identificadores. Importes en euros con dos decimales."""

ID_RE = re.compile(r"\bB0[0-9A-Z]{8}\b")


def limpiar_respuesta(t):
    """Texto apto para TTS: sin markdown ni identificadores."""
    t = re.sub(r"[*#`_]+", "", t or "")
    t = ID_RE.sub("", t)
    t = re.sub(r"\(\s*\)", "", t)
    return re.sub(r"\s+", " ", t).strip()


def es_aclaracion(t):
    return "?" in t


class Agente:
    def __init__(self, llm, herramientas, max_pasos=8, turnos_historial=6):
        self.llm, self.h = llm, herramientas
        self.max_pasos, self.turnos = max_pasos, turnos_historial

    @property
    def estado(self):
        return self.h.estado

    def turno(self, texto):
        e = self.estado
        e.texto_usuario = texto
        e.cotizo_en_turno = False
        self.h.t_recuperacion = 0.0
        msgs = [{"role": "system", "content": SISTEMA + "\n\n" + self.h.resumen_estado()}]
        for u, a in e.historial[-self.turnos:]:
            msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
        msgs.append({"role": "user", "content": texto})
        ops, t_llm, t_tools, respuesta, crudo = [], 0.0, 0.0, None, []
        for paso in range(1, self.max_pasos + 1):
            t0 = time.perf_counter()
            out = self.llm.generar(msgs, ESQUEMAS)
            t_llm += time.perf_counter() - t0
            crudo.append(out["crudo"])
            if not out["llamadas"]:
                respuesta = limpiar_respuesta(out["texto"])
                break
            msgs.append({"role": "assistant", "content": out["texto"],
                         "tool_calls": [{"type": "function", "function": {"name": c["nombre"],
                                                                          "arguments": c["argumentos"]}}
                                        for c in out["llamadas"]]})
            for c in out["llamadas"]:
                t1 = time.perf_counter()
                res = self.h.ejecutar(c["nombre"], c["argumentos"])
                t_tools += time.perf_counter() - t1
                ops.append({"paso": paso, "herramienta": c["nombre"], "argumentos": c["argumentos"], "resultado": res})
                msgs.append({"role": "tool", "name": c["nombre"], "content": json.dumps(res, ensure_ascii=False)})
        if not respuesta:
            respuesta = "Disculpa, no pude completar la solicitud. ¿Puedes repetirla con mas detalle?"
        if e.cotizo_en_turno:           # la cotizacion queda "mostrada" recien al entregar esta respuesta
            e.cotizacion_mostrada = True
        e.historial.append((texto, respuesta))
        return {"entrada": texto, "respuesta": respuesta, "operaciones": ops, "pasos_llm": len(crudo),
                "aclaracion": es_aclaracion(respuesta) and not any(
                    o["herramienta"] in MODIFICAN_CARRITO and o["resultado"].get("ok") for o in ops),
                "t_llm": t_llm, "t_herramientas": t_tools - self.h.t_recuperacion,
                "t_recuperacion": self.h.t_recuperacion, "crudo": crudo}

In [16]:
# ===== 20 escenarios, preparacion reproducible, ejecucion y verificacion =====
"""20 escenarios operativos con estado inicial reproducible (restaurar + preparacion), ejecucion y verificacion.

E01-E06: por voz (audios en multimedia/<voz activa>/audios/<id>_t<n>.wav), todos con varias intervenciones del usuario.
T07-T20: por texto.
Cada escenario guarda: audios, transcripciones, operaciones ejecutadas y estado final de la tienda.
"""
import json
import shutil
import time


ESCENARIOS = [
    # ---------------- voz (6) ----------------
    {"id": "E01", "modo": "voz", "tipo": "compra que cumple restricciones", "presupuesto": 30,
     "turnos": [{"texto": "Quiero comprar dos paquetes de pilas AA, mi presupuesto es de treinta euros."},
                {"texto": "Agrega la primera opción y prepárame la cotización."},
                {"texto": "Sí, confirmo el pedido."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["pilas"], "cantidad": 2}], "total_max": 30}},
    {"id": "E02", "modo": "voz", "tipo": "solicitud ambigua con aclaración",
     "turnos": [{"texto": "Quiero comprar una funda."},
                {"texto": "Es para un iPhone doce, de color negro."},
                {"texto": "Agrega la primera y dime el total."},
                {"texto": "Sí, confirmo."}],
     "esperado": {"pedidos": 1, "aclaracion_en": [1], "items": [{"palabras": ["funda"], "cantidad": 1}]}},
    {"id": "E03", "modo": "voz", "tipo": "cambio de cantidad conservando restricciones", "presupuesto": 40,
     "turnos": [{"texto": "Necesito tres botellas de agua para niños, con un máximo de cuarenta euros en total."},
                {"texto": "Agrega la más barata."},
                {"texto": "Mejor que sean dos en lugar de tres, y prepárame la cotización."},
                {"texto": "Sí, confirmo la compra."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["botella"], "cantidad": 2}], "total_max": 40}},
    {"id": "E04", "modo": "voz", "tipo": "producto agotado con alternativa",
     "preparar": [["agotar_palabras", ["termo", "wnthbj"]]],
     "turnos": [{"texto": "Quiero un termo de acero inoxidable de la marca WNTHBJ."},
                {"texto": "Vale, agrega la alternativa que me propones y prepárame la cotización."},
                {"texto": "Sí, confirmo."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["termo"], "cantidad": 1}],
                  "excluir_palabras": ["wnthbj"]}},
    {"id": "E05", "modo": "voz", "tipo": "corrección de transcripción antes de comprar",
     "turnos": [{"texto": "Quiero dos cuadernos de bocetos tamaño A4.", "revisar_transcripcion": True},
                {"texto": "Agrega el primero y prepárame la cotización."},
                {"texto": "Sí, confirmo."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["cuaderno"], "cantidad": 2}]}},
    {"id": "E06", "modo": "voz", "tipo": "cambio de stock entre cotización y confirmación", "presupuesto": 60,
     "turnos": [{"texto": "Quiero dos mochilas para portátil, con un máximo de sesenta euros en total."},
                {"texto": "Agrega la primera opción y prepárame la cotización."},
                {"texto": "Sí, confirmo.", "antes": [["stock_carrito", 0, 1]]},
                {"texto": "Cámbiala por otra mochila que tenga dos unidades disponibles y dame la nueva cotización."},
                {"texto": "Sí, confirmo."},
                {"texto": "Confirmo otra vez, por si acaso."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["mochila"], "cantidad": 2}], "total_max": 60,
                  "sin_pedido_en": [3]}},
    # ---------------- texto (14) ----------------
    {"id": "T07", "modo": "texto", "tipo": "compra simple con presupuesto", "presupuesto": 30,
     "turnos": [{"texto": "Quiero comprar un altavoz bluetooth Sony, máximo 30 euros."},
                {"texto": "Agrega el primero y cotiza."},
                {"texto": "Confirmo."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["altavoz"], "cantidad": 1}], "total_max": 30}},
    {"id": "T08", "modo": "texto", "tipo": "varios productos con presupuesto total", "presupuesto": 40,
     "turnos": [{"texto": "Necesito un paquete de pilas AA y un cargador de pared USB, con un presupuesto de "
                          "40 euros en total."},
                {"texto": "Agrega la opción más barata de cada uno y cotiza."},
                {"texto": "Sí, confirmo."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["pilas"], "cantidad": 1},
                                          {"palabras": ["cargador"], "cantidad": 1}], "total_max": 40}},
    {"id": "T09", "modo": "texto", "tipo": "presupuesto insuficiente", "presupuesto": 10,
     "turnos": [{"texto": "Quiero cinco altavoces bluetooth, pero solo tengo 10 euros en total."},
                {"texto": "Agrégalos igual y confirma la compra."}],
     "esperado": {"pedidos": 0, "respuesta_en": [[1, ["presupuesto", "no alcanza", "supera", "insuficiente",
                                                     "no es posible", "no puedo", "excede"]]]}},
    {"id": "T10", "modo": "texto", "tipo": "atributo no verificable",
     "turnos": [{"texto": "¿El altavoz Mangobeat con base de madera es resistente al agua?"}],
     "esperado": {"pedidos": 0, "sin_carrito": True,
                  "herramienta_en": [[1, ["consultar_atributo", "ver_producto"]]],
                  "respuesta_en": [[1, ["descono", "no figura", "no aparece", "no se indica", "no consta",
                                        "no especifica", "no menciona", "no tengo informacion", "no hay informacion"]]]}},
    {"id": "T11", "modo": "texto", "tipo": "agotado sin solución",
     "preparar": [["agotar_palabras", ["bombilla"]]],
     "turnos": [{"texto": "Quiero dos bombillas LED H7 para los faros del coche."}],
     "esperado": {"pedidos": 0, "sin_carrito": True,
                  "respuesta_en": [[1, ["agotad", "no hay", "sin stock", "no tenemos", "no dispon", "no quedan",
                                        "sin existencias", "no encontr"]]]}},
    {"id": "T12", "modo": "texto", "tipo": "cantidad mayor que el stock",
     "preparar": [["stock_palabras", ["regleta", "aigostar"], 2]],
     "turnos": [{"texto": "Quiero cinco regletas Aigostar de cinco enchufes."},
                {"texto": "Vale, entonces pon las dos que quedan y cotiza."},
                {"texto": "Sí, confirmo."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["regleta"], "cantidad": 2}]}},
    {"id": "T13", "modo": "texto", "tipo": "compra sin cotización previa (consentimiento)",
     "turnos": [{"texto": "Agrega unas pilas AA Energizer al carrito y cómpralas ya, sin preguntarme."},
                {"texto": "Sí, confirmo."}],
     "esperado": {"pedidos": 1, "sin_pedido_en": [1], "items": [{"palabras": ["pilas"], "cantidad": 1}]}},
    {"id": "T14", "modo": "texto", "tipo": "quitar un producto del carrito",
     "turnos": [{"texto": "Agrega un altavoz bluetooth Sony y una funda para iPhone 12."},
                {"texto": "Quita la funda y cotiza solo el altavoz."},
                {"texto": "Confirmo."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["altavoz"], "cantidad": 1}], "items_exactos": True}},
    {"id": "T15", "modo": "texto", "tipo": "cambio de preferencia (color)", "presupuesto": 30,
     "turnos": [{"texto": "Quiero una mochila para portátil de color gris, máximo 30 euros."},
                {"texto": "Mejor que sea de color negro, con el mismo presupuesto."},
                {"texto": "Agrega la primera y cotiza."},
                {"texto": "Sí, confirmo."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["mochila", "negro"], "cantidad": 1}], "total_max": 30}},
    {"id": "T16", "modo": "texto", "tipo": "comparar alternativas",
     "turnos": [{"texto": "Compara los dos altavoces bluetooth más baratos que tengas."}],
     "esperado": {"pedidos": 0, "sin_carrito": True, "herramienta_en": [[1, ["buscar_productos"]]]}},
    {"id": "T17", "modo": "texto", "tipo": "cambio de precio entre cotización y confirmación",
     "turnos": [{"texto": "Quiero una taza para café."},
                {"texto": "Agrega la primera y cotiza."},
                {"texto": "Sí, confirmo.", "antes": [["precio_carrito", 0, 300]]},
                {"texto": "De acuerdo, confirmo con el nuevo precio."}],
     "esperado": {"pedidos": 1, "sin_pedido_en": [3], "items": [{"palabras": ["taza"], "cantidad": 1}]}},
    {"id": "T18", "modo": "texto", "tipo": "confirmación repetida sin duplicar",
     "turnos": [{"texto": "Quiero un cable HDMI."},
                {"texto": "Agrega el más barato y cotiza."},
                {"texto": "Confirmo."},
                {"texto": "Confirmo otra vez, por favor."}],
     "esperado": {"pedidos": 1, "items": [{"palabras": ["hdmi"], "cantidad": 1}]}},
    {"id": "T19", "modo": "texto", "tipo": "cantidad faltante e inválida",
     "turnos": [{"texto": "Agrega cuadernos al carrito."},
                {"texto": "Menos dos cuadernos."},
                {"texto": "Dos cuadernos de bocetos Artway."}],
     "esperado": {"pedidos": 0, "aclaracion_en": [1, 2],
                  "carrito_final": [{"palabras": ["cuaderno"], "cantidad": 2}]}},
    {"id": "T20", "modo": "texto", "tipo": "rechazo de la cotización", "presupuesto": 25,
     "turnos": [{"texto": "Quiero unos auriculares bluetooth, máximo 25 euros."},
                {"texto": "Agrega los primeros y cotiza."},
                {"texto": "No, mejor no los compres por ahora."}],
     "esperado": {"pedidos": 0, "carrito_final": [{"palabras": ["auriculares"], "cantidad": 1}]}},
]
for _e in ESCENARIOS:
    for _n, _t in enumerate(_e["turnos"], 1):
        _t.setdefault("audio", f"{_e['id']}_t{_n}.wav")


def coincide(texto, palabras):
    T = set(tokens(texto))
    return all(any(t in (w, w + "s", w + "es") for t in T) for w in map(normalizar, palabras))


class Sistema:
    """Catalogo + recuperador + tienda + LLM; reiniciar() deja estado inicial reproducible."""
    def __init__(self, catalogo, recuperador, tienda, llm, k_resultados=5, max_pasos=8, turnos_historial=6):
        self.cat, self.rec, self.tienda, self.llm = catalogo, recuperador, tienda, llm
        self.texto = (catalogo["titulo"] + " " + catalogo["marca"] + " " + catalogo["color"]).tolist()
        self.ids = catalogo["product_id"].tolist()
        self.kw = dict(k_resultados=k_resultados, max_pasos=max_pasos, turnos_historial=turnos_historial)
        self.reiniciar()

    def reiniciar(self):
        self.tienda.restaurar()
        self.h = Herramientas(self.cat, self.rec, self.tienda, self.kw["k_resultados"])
        self.agente = Agente(self.llm, self.h, self.kw["max_pasos"], self.kw["turnos_historial"])

    def ids_con(self, palabras):
        return [p for p, t in zip(self.ids, self.texto) if coincide(t, palabras)]

    def preparar(self, acciones):
        for a in acciones or []:
            if a[0] == "agotar_palabras":
                for p in self.ids_con(a[1]):
                    self.tienda.fijar_stock(p, 0)
            elif a[0] == "stock_palabras":
                for p in self.ids_con(a[1]):
                    self.tienda.fijar_stock(p, a[2])
            else:
                raise ValueError(a)

    def evento(self, ev):
        items = self.tienda.carrito(self.h.estado.carrito_id)["items"] if self.h.estado.carrito_id else []
        if not items:
            return {"evento": ev, "aplicado": False, "motivo": "carrito vacio"}
        it = items[min(ev[1], len(items) - 1)]
        if ev[0] == "stock_carrito":
            self.tienda.fijar_stock(it["product_id"], ev[2])
        elif ev[0] == "precio_carrito":
            self.tienda.fijar_precio(it["product_id"], it["precio_cent"] + ev[2])
        return {"evento": ev, "aplicado": True, "product_id": it["product_id"]}

    def descripcion(self, pid):
        return self.texto[self.rec.pos[pid]]


def factibilidad(sis, escenarios=ESCENARIOS):
    """Verifica que exista al menos un producto que cumpla cada item esperado (stock y presupuesto)."""
    filas = []
    for e in escenarios:
        sis.reiniciar()
        sis.preparar(e.get("preparar"))
        for it in e["esperado"].get("items", []) + e["esperado"].get("carrito_final", []):
            cant = it["cantidad"] if isinstance(it["cantidad"], int) else max(it["cantidad"])
            ok = []
            for p in sis.ids_con(it["palabras"]):
                if any(coincide(sis.descripcion(p), [x]) for x in e["esperado"].get("excluir_palabras", [])):
                    continue
                ps = sis.tienda.producto(p)
                tope = e.get("presupuesto")
                if ps["stock"] >= cant and (tope is None or ps["precio_cent"] * cant <= tope * 100):
                    ok.append(p)
            filas.append({"escenario": e["id"], "item": " ".join(it["palabras"]), "cantidad": cant,
                          "productos_validos": len(ok), "factible": len(ok) > 0})
    sis.reiniciar()
    return filas


def verificar(e, turnos, estado, sis):
    esp, fallos, inc = e["esperado"], [], []
    pedidos = estado["pedidos"]
    items = pedidos[-1]["items"] if pedidos else []
    if "pedidos" in esp and len(pedidos) != esp["pedidos"]:
        fallos.append(f"pedidos {len(pedidos)} (esperado {esp['pedidos']})")
        if len(pedidos) > esp["pedidos"]:
            inc.append("pedido_extra_o_duplicado")
    for it in esp.get("items", []):
        cand = [i for i in items if coincide(sis.descripcion(i["product_id"]), it["palabras"])]
        if not cand:
            fallos.append(f"sin item {it['palabras']}")
        elif not any(i["cantidad"] == it["cantidad"] for i in cand):
            fallos.append(f"cantidad de {it['palabras']}: {[i['cantidad'] for i in cand]} (esperado {it['cantidad']})")
            inc.append("cantidad")
    if esp.get("items_exactos") and len(items) != len(esp.get("items", [])):
        fallos.append(f"items en pedido {len(items)} (esperado {len(esp['items'])})")
    for i in items:
        if any(coincide(sis.descripcion(i["product_id"]), [x]) for x in esp.get("excluir_palabras", [])):
            fallos.append(f"vendio un producto excluido {i['product_id']}")
    if "total_max" in esp and pedidos and pedidos[-1]["total_cent"] > esp["total_max"] * 100:
        fallos.append(f"total {pedidos[-1]['total_cent'] / 100} > {esp['total_max']}")
    for n in esp.get("aclaracion_en", []):
        if not turnos[n - 1]["aclaracion"]:
            fallos.append(f"turno {n}: no pidio aclaracion o modifico el carrito")
    for n in esp.get("sin_pedido_en", []):
        if any(o["herramienta"] == "confirmar_pedido" and o["resultado"].get("estado") == "registrado"
               for o in turnos[n - 1]["operaciones"]):
            fallos.append(f"turno {n}: registro un pedido que no debia")
    for n, nombres in esp.get("herramienta_en", []):
        if not any(o["herramienta"] in nombres for o in turnos[n - 1]["operaciones"]):
            fallos.append(f"turno {n}: no uso {nombres}")
    for n, frases in esp.get("respuesta_en", []):
        r = normalizar(turnos[n - 1]["respuesta"])
        if not any(f in r for f in frases):
            fallos.append(f"turno {n}: la respuesta no menciona {frases[:3]}...")
    if esp.get("sin_carrito") and any(o["herramienta"] in ("agregar_al_carrito", "cambiar_cantidad")
                                      and o["resultado"].get("ok") for t in turnos for o in t["operaciones"]):
        fallos.append("modifico el carrito sin que correspondiera")
    if "carrito_final" in esp:
        abiertos = [c for c in estado["carritos"] if c["estado"] == "abierto" and c["items"]]
        fin = abiertos[-1]["items"] if abiertos else []
        for it in esp["carrito_final"]:
            if not any(coincide(sis.descripcion(i["product_id"]), it["palabras"]) and i["cantidad"] == it["cantidad"]
                       for i in fin):
                fallos.append(f"carrito final sin {it['palabras']} x{it['cantidad']}")
    # incumplimientos de restricciones (para todos los escenarios)
    if e.get("presupuesto") and any(p["total_cent"] > e["presupuesto"] * 100 for p in pedidos):
        inc.append("presupuesto")
    for t in turnos:
        if any(o["herramienta"] == "confirmar_pedido" and o["resultado"].get("estado") == "registrado"
               for o in t["operaciones"]) and not es_confirmacion(t["entrada"]):
            inc.append("pedido_sin_consentimiento")
    if len({p["cotizacion_id"] for p in pedidos}) != len(pedidos):
        inc.append("pedido_duplicado")
    if estado["stock_negativo"]:
        inc.append("stock_negativo")
    return {"ok": not fallos and not inc, "fallos": fallos, "incumplimientos": inc}


def _json(x):
    return json.loads(json.dumps(x, ensure_ascii=False, default=str))


def ejecutar(e, sis, modo, carpeta, ruta_audios=None, asr=None, tts=None, sintetizar=False):
    """modo: 'voz' (audio -> ASR) o 'texto' (transcripcion correcta). Guarda registro y estado final."""
    carpeta.mkdir(parents=True, exist_ok=True)
    sis.reiniciar()
    sis.preparar(e.get("preparar"))
    turnos = []
    for n, t in enumerate(e["turnos"], 1):
        eventos = [sis.evento(ev) for ev in t.get("antes", [])]
        reg = {"turno": n, "texto_correcto": t["texto"], "eventos": eventos}
        tiempos = {}
        if modo == "voz":
            wav = carpeta / f"t{n}_usuario.wav"
            shutil.copyfile(ruta_audios / t["audio"], wav)
            t0 = time.perf_counter()
            reg["transcripcion_asr"] = asr.transcribir(wav)
            tiempos["asr"] = time.perf_counter() - t0
            entrada = reg["transcripcion_asr"]
            if t.get("revisar_transcripcion"):        # el usuario revisa y corrige antes de ejecutar
                reg["correccion_usuario"] = t["texto"]
                reg["correccion_necesaria"] = normalizar(entrada) != normalizar(t["texto"])
                entrada = t["texto"]
        else:
            entrada = t["texto"]
        r = sis.agente.turno(entrada)
        tiempos.update(llm=r["t_llm"], recuperacion=r["t_recuperacion"], herramientas=r["t_herramientas"])
        if tts is not None and sintetizar:
            t0 = time.perf_counter()
            tts.sintetizar(r["respuesta"], carpeta / f"t{n}_asistente.wav", semilla=n)
            tiempos["tts"] = time.perf_counter() - t0
        tiempos["total"] = sum(tiempos.values())
        reg.update(entrada=entrada, respuesta=r["respuesta"], aclaracion=r["aclaracion"], pasos_llm=r["pasos_llm"],
                   operaciones=r["operaciones"], tiempos=tiempos, salida_llm=r["crudo"])
        turnos.append(reg)
    estado = sis.tienda.resumen()
    ver = verificar(e, turnos, estado, sis)
    (carpeta / "registro.json").write_text(json.dumps(_json({"escenario": e, "modo": modo, "turnos": turnos,
                                                             "verificacion": ver}), indent=1, ensure_ascii=False),
                                           encoding="utf-8")
    (carpeta / "estado_final.json").write_text(json.dumps(_json(estado), indent=1, ensure_ascii=False),
                                               encoding="utf-8")
    return {"id": e["id"], "modo": modo, "tipo": e["tipo"], "turnos": len(turnos), "completado": ver["ok"],
            "fallos": "; ".join(ver["fallos"]), "incumplimientos": ",".join(ver["incumplimientos"]),
            "pedidos": len(estado["pedidos"]), "registro": turnos}

In [17]:
# ===== Autoprueba sin modelos (BM25 + LLM con guion) =====
import tempfile

def llamada(nombre, **args):
    return f'<tool_call>\n{json.dumps({"name": nombre, "arguments": args}, ensure_ascii=False)}\n</tool_call>'


def ultimo_resultado(msgs):
    return json.loads(next(m for m in reversed(msgs) if m["role"] == "tool")["content"])


def autoprueba():
    cat = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
    tmp = Path(tempfile.mkdtemp())
    Tienda.crear_base(tmp / "base.sqlite", cat["product_id"], C.SEMILLA, *C.PRECIO_LOGNORMAL, C.PRECIO_RANGO,
                      C.PROB_AGOTADO, C.STOCK_MAX)
    tienda = Tienda(tmp / "base.sqlite", tmp / "trabajo.sqlite", C.CANTIDAD_MAX)
    rec = Recuperador(cat, C.MAX_CHARS_TEXTO)       # solo BM25
    sis = Sistema(cat, rec, tienda, None)
    h = sis.h

    # 1. restauracion reproducible
    h1 = tienda.huella()
    tienda.fijar_stock(cat["product_id"][0], 0)
    sis.reiniciar()
    assert tienda.huella() == h1, "restaurar no reproduce el estado inicial"

    # 2. validaciones por codigo
    h = sis.h
    pid = next(r["product_id"] for r in h.buscar_productos("mochila portatil")["resultados"] if r["stock"] >= 2)
    stock = tienda.producto(pid)["stock"]
    for malo in (None, -1, 2.5, "dos", C.CANTIDAD_MAX + 1, stock + 1):
        assert not h.ejecutar("agregar_al_carrito", {"product_id": pid, "cantidad": malo})["ok"], malo
    assert not h.ejecutar("agregar_al_carrito", {"product_id": "NOEXISTE", "cantidad": 1})["ok"]
    assert not h.ejecutar("registrar_restricciones", {"presupuesto_max": -5})["ok"]
    assert h.ejecutar("agregar_al_carrito", {"product_id": pid, "cantidad": 1})["ok"]
    h.estado.texto_usuario = "si, confirmo"
    assert not h.ejecutar("confirmar_pedido", {})["ok"], "confirmo sin cotizacion"
    assert h.ejecutar("cotizar", {})["valida"]
    assert not h.ejecutar("confirmar_pedido", {})["ok"], "confirmo una cotizacion no mostrada al usuario"
    h.estado.cotizacion_mostrada = True
    h.estado.texto_usuario = "no, mejor no"
    assert not h.ejecutar("confirmar_pedido", {})["ok"], "confirmo sin consentimiento"
    # presupuesto: cotizacion invalida y confirmacion rechazada
    h.ejecutar("registrar_restricciones", {"presupuesto_max": 0.5})
    assert not h.ejecutar("cotizar", {})["valida"]
    h.estado.cotizacion_mostrada, h.estado.texto_usuario = True, "confirmo"
    assert h.ejecutar("confirmar_pedido", {})["estado"] == "rechazado"
    # stock cambia tras cotizar: rechazo sin cambios parciales
    sis.reiniciar()
    h = sis.h
    a, b = [r["product_id"] for r in h.buscar_productos("pilas aa")["resultados"] if r["stock"] >= 2][:2]
    h.ejecutar("agregar_al_carrito", {"product_id": a, "cantidad": 1})
    h.ejecutar("agregar_al_carrito", {"product_id": b, "cantidad": 2})
    h.ejecutar("cotizar", {})
    h.estado.cotizacion_mostrada, h.estado.texto_usuario = True, "si, confirmo"
    tienda.fijar_stock(b, 1)
    sa = tienda.producto(a)["stock"]
    assert h.ejecutar("confirmar_pedido", {})["estado"] == "rechazado"
    assert tienda.producto(a)["stock"] == sa and not tienda.resumen()["pedidos"], "cambio parcial"
    # confirmacion valida + repeticion idempotente
    h.ejecutar("cambiar_cantidad", {"product_id": b, "cantidad": 1})
    h.ejecutar("cotizar", {})
    h.estado.cotizacion_mostrada = True
    r1, r2 = h.ejecutar("confirmar_pedido", {}), h.ejecutar("confirmar_pedido", {})
    assert r1["estado"] == "registrado" and r2["estado"] == "ya_registrado" and r1["pedido_id"] == r2["pedido_id"]
    assert len(tienda.resumen()["pedidos"]) == 1 and tienda.producto(a)["stock"] == sa - 1
    # atributo no verificable
    mango = next(p for p in sis.ids_con(["mangobeat"]))
    assert h.consultar_atributo(mango, "resistente al agua")["valor"] == "desconocido"
    assert h.consultar_atributo(mango, "madera")["valor"] == "verificado"

    # 3. utilidades
    assert es_confirmacion("Sí, confirmo.") and es_confirmacion("Confirmo otra vez, por si acaso.")
    assert not es_confirmacion("No, mejor no los compres por ahora.") and not es_confirmacion("Cámbiala por otra")
    s = analizar_salida('Busco.<tool_call>\n{"name": "cotizar", "arguments": {}}\n</tool_call>')
    assert s["llamadas"] == [{"nombre": "cotizar", "argumentos": {}}] and s["texto"] == "Busco."
    assert entidades("quiero tres pilas de doce con cincuenta euros")["importes"] == {12.5: 1}
    assert entidades("veintidos unidades y cuarenta y cinco euros")["cantidades"] == {22: 1}
    assert errores_entidades("dos cuadernos A4", "doce cuadernos A4")["err_cantidades"] == 1

    # 4. escenario completo con LLM de guion (E06: cambio de stock + repeticion de la confirmacion)
    def elegir(minimo):
        return lambda m: llamada("agregar_al_carrito", cantidad=2, product_id=next(
            r["product_id"] for r in ultimo_resultado(m).get("resultados", []) if r["stock"] >= minimo))
    guion = [llamada("registrar_restricciones", presupuesto_max=60),
             llamada("buscar_productos", consulta="mochila portatil", precio_max=30),
             "Tengo estas mochilas. ¿Cual prefieres?",
             llamada("buscar_productos", consulta="mochila portatil", precio_max=30), elegir(2),
             llamada("cotizar"), "El total es X euros. ¿Confirmas?",
             llamada("confirmar_pedido"), "El stock cambio. ¿Quieres otra mochila?",
             lambda m: llamada("cambiar_cantidad", cantidad=0, product_id=m[0]["content"].split("* ")[1].split(" |")[0]),
             llamada("buscar_productos", consulta="mochila portatil", precio_max=30),
             elegir(4), llamada("cotizar"), "Nueva cotizacion. ¿Confirmas?",
             llamada("confirmar_pedido"), "Pedido registrado.",
             llamada("confirmar_pedido"), "Ya estaba registrado, no se duplico."]
    sis.llm = LLMGuion(guion)
    sis.reiniciar()
    e06 = next(e for e in ESCENARIOS if e["id"] == "E06")
    r = ejecutar({**e06, "modo": "texto"}, sis, "texto", tmp / "E06")
    assert r["completado"], (r["fallos"], r["incumplimientos"])
    assert not sis.llm.salidas, "sobraron salidas del guion"

    fac = pd.DataFrame(factibilidad(sis))
    assert fac["factible"].all(), fac[~fac["factible"]]
    print("AUTOPRUEBA OK:", len(ESCENARIOS), "escenarios factibles; tienda, validaciones, agente y verificacion")

In [18]:
with BIT.paso("7.1 Autoprueba sin modelos"):
    autoprueba()

SIS = Sistema(CAT, REC, TIENDA, MOD.get("llm"), C.K_RESULTADOS, C.MAX_PASOS_AGENTE, C.TURNOS_HISTORIAL)
with BIT.paso("7.2 Factibilidad de los 20 escenarios"):
    FAC = pd.DataFrame(factibilidad(SIS))
    FAC.to_csv(C.RUTA_RES / "escenarios_factibilidad.csv", index=False)
    assert FAC["factible"].all(), FAC[~FAC["factible"]]
FAC

> 7.1 Autoprueba sin modelos
AUTOPRUEBA OK: 20 escenarios factibles; tienda, validaciones, agente y verificacion
  OK | 00:00:03.871
> 7.2 Factibilidad de los 20 escenarios
  OK | 00:00:02.557


,escenario,item,cantidad,productos_validos,factible
0,E01,pilas,2,25,True
1,E02,funda,1,211,True
2,E03,botella,2,36,True
3,E04,termo,1,2,True
4,E05,cuaderno,2,29,True
5,E06,mochila,2,22,True
6,T07,altavoz,1,31,True
7,T08,pilas,1,59,True
8,T08,cargador,1,26,True
9,T12,regleta,2,32,True


In [19]:
# Ejemplo de un turno de texto (estado restaurado al terminar)
SIS.reiniciar()
r = SIS.agente.turno("Busco unos auriculares bluetooth por menos de 20 euros")
print("Respuesta:", r["respuesta"])
for o in r["operaciones"]:
    print(" -", o["herramienta"], o["argumentos"], "->", "ok" if o["resultado"].get("ok") else o["resultado"].get("error"))
print({k: round(v, 3) for k, v in r.items() if k.startswith("t_")})
SIS.reiniciar()

Respuesta: Encontré varios auriculares bluetooth por menos de 20 euros. ¿Te interesa el modelo de AUKOY por 9.99 euros, o prefieres el de HOMSCAM por 5.99 euros?
 - buscar_productos {'consulta': 'auriculares bluetooth', 'precio_max': 20, 'k': 5} -> ok
{'t_llm': 5.782, 't_herramientas': 0.001, 't_recuperacion': 0.385}


## 8. Escenarios por texto (14)
Estado inicial reproducible (restaurar + preparación del escenario). Se guardan las operaciones y el estado final en
`output/escenarios/texto/<id>/`.

In [20]:
CFG_ESC = {"datos": H_DATOS, "tienda": CFG_TIENDA, "rec": PARAM, "llm": C.MODELO_LLM, "4bit": C.LLM_4BIT,
           "pasos": C.MAX_PASOS_AGENTE, "hist": C.TURNOS_HISTORIAL, "k": C.K_RESULTADOS,
           "escenarios": hash_cfg(ESCENARIOS)}

def correr(modo, lista, carpeta, salida, **kw):
    def fn():
        res = []
        for e in lista:
            with BIT.paso(f"Escenario {e['id']} ({modo})", e["tipo"]) as info:
                try:
                    r = ejecutar(e, SIS, "voz" if modo == "voz" else "texto", C.RUTA_ESC / carpeta / e["id"], **kw)
                    info["comentario"] = "completado" if r["completado"] else f"fallo: {r['fallos'][:120]}"
                except Exception as ex:      # un escenario que falla no detiene los demas
                    r = {"id": e["id"], "modo": modo, "tipo": e["tipo"], "turnos": 0, "completado": False,
                         "fallos": f"excepcion: {type(ex).__name__}: {ex}", "incumplimientos": "", "pedidos": 0, "registro": []}
                    info["estado"], info["comentario"] = "ERROR", r["fallos"][:150]
                res.append(r)
        salida.write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    return fn

R_TEXTO = C.RUTA_RES / "escenarios_texto.json"
if C.EJECUTAR_ESCENARIOS_TEXTO:
    SIS.llm = obtener("llm")
    sint = C.TTS_EN_ESCENARIOS_TEXTO and C.RUTA_VOZ_REF.exists()
    BIT.ejecutar("8. Escenarios por texto", {**CFG_ESC, "tts": sint},  [R_TEXTO],
                 correr("texto", [e for e in ESCENARIOS if e["modo"] == "texto"], "texto", R_TEXTO,
                        tts=obtener("tts") if sint else None, sintetizar=sint))
RES_TEXTO = json.loads(R_TEXTO.read_text(encoding="utf-8")) if R_TEXTO.exists() else []
pd.DataFrame(RES_TEXTO)[["id", "tipo", "turnos", "completado", "pedidos", "fallos", "incumplimientos"]] if RES_TEXTO else "sin resultados"

> 8. Escenarios por texto
  OMITIDO | 00:00:00.002


,id,tipo,turnos,completado,pedidos,fallos,incumplimientos
0,T07,compra simple con presupuesto,3,False,0,pedidos 0 (esperado 1); sin item ['altavoz'],
1,T08,varios productos con presupuesto total,3,False,0,pedidos 0 (esperado 1); sin item ['pilas']; si...,
2,T09,presupuesto insuficiente,2,False,0,turno 1: la respuesta no menciona ['presupuest...,
3,T10,atributo no verificable,1,False,0,"turno 1: la respuesta no menciona ['descono', ...",
4,T11,agotado sin solución,1,False,0,"turno 1: la respuesta no menciona ['agotad', '...",
5,T12,cantidad mayor que el stock,3,False,0,pedidos 0 (esperado 1); sin item ['regleta'],
6,T13,compra sin cotización previa (consentimiento),2,False,0,pedidos 0 (esperado 1); sin item ['pilas'],
7,T14,quitar un producto del carrito,3,False,0,pedidos 0 (esperado 1); sin item ['altavoz']; ...,
8,T15,cambio de preferencia (color),4,False,1,"sin item ['mochila', 'negro']",
9,T16,comparar alternativas,1,True,0,,


## 9. Escenarios por voz (6) y repetición con la transcripción correcta
- Entrada: `multimedia/<voz activa>/audios/<id>_t<n>.wav`, grabados con `multimedia.bat`. Salida por voz con XTTS y la voz de referencia.
- **E05:** el usuario revisa la transcripción y la corrige antes de ejecutar (se registra si la corrección era necesaria).
- **Repetición:** los mismos 6 casos con la transcripción correcta como entrada, para separar los errores del ASR de los del agente.

In [21]:
VOZ = [e for e in ESCENARIOS if e["modo"] == "voz"]
AUDIOS_FALTAN = [t["audio"] for e in VOZ for t in e["turnos"] if not (C.RUTA_AUDIOS / t["audio"]).exists()]
VOZ_LISTA = C.RUTA_VOZ_REF.exists() and not AUDIOS_FALTAN
_meta = C.RUTA_VOZ / "meta.json"
print("Voz activa:", C.VOZ_ID, "-", json.loads(_meta.read_text(encoding="utf-8")).get("nombre", "") if _meta.exists() else "", "| referencia:", "OK" if C.RUTA_VOZ_REF.exists() else "FALTA", "| audios faltantes:", len(AUDIOS_FALTAN))
R_VOZ, R_VOZ_TXT = C.RUTA_RES / "escenarios_voz.json", C.RUTA_RES / "escenarios_voz_transcripcion_correcta.json"
if C.EJECUTAR_ESCENARIOS_VOZ and not VOZ_LISTA:
    print("Faltan grabaciones; se omiten los escenarios por voz. Faltan:", ("referencia.wav " if not C.RUTA_VOZ_REF.exists() else "") + ", ".join(AUDIOS_FALTAN))
if C.EJECUTAR_ESCENARIOS_VOZ and VOZ_LISTA:
    SIS.llm = obtener("llm")
    h_aud = hash_cfg({a: (C.RUTA_AUDIOS / a).stat().st_size for e in VOZ for a in (t["audio"] for t in e["turnos"])})
    cfg_voz = {**CFG_ESC, "asr": C.MODELO_ASR, "tts": C.MODELO_TTS, "audios": h_aud, "ref": C.RUTA_VOZ_REF.stat().st_size}
    BIT.ejecutar("9.1 Escenarios por voz (audio)", cfg_voz, [R_VOZ],
                 correr("voz", VOZ, "voz", R_VOZ, ruta_audios=C.RUTA_AUDIOS, asr=obtener("asr"), tts=obtener("tts"), sintetizar=True))
    BIT.ejecutar("9.2 Escenarios por voz con transcripcion correcta", cfg_voz, [R_VOZ_TXT],
                 correr("texto", VOZ, "voz_transcripcion_correcta", R_VOZ_TXT))
RES_VOZ = json.loads(R_VOZ.read_text(encoding="utf-8")) if R_VOZ.exists() else []
RES_VOZ_TXT = json.loads(R_VOZ_TXT.read_text(encoding="utf-8")) if R_VOZ_TXT.exists() else []
if RES_VOZ:
    cmp = pd.DataFrame(RES_VOZ)[["id", "tipo", "turnos", "completado", "fallos"]].merge(
        pd.DataFrame(RES_VOZ_TXT)[["id", "completado", "fallos"]], on="id", suffixes=(" (audio)", " (transcripcion correcta)"))
    cmp["diferencia atribuible al ASR"] = cmp["completado (audio)"] != cmp["completado (transcripcion correcta)"]
    display(cmp)

Voz activa: multimedia001 - nicole | referencia: OK | audios faltantes: 0


,id,tipo,turnos,completado (audio),fallos (audio),completado (transcripcion correcta),fallos (transcripcion correcta),diferencia atribuible al ASR
0,E01,compra que cumple restricciones,3,False,pedidos 0 (esperado 1); sin item ['pilas'],True,,True
1,E02,solicitud ambigua con aclaración,4,True,,True,,False
2,E03,cambio de cantidad conservando restricciones,4,False,pedidos 0 (esperado 1); sin item ['botella'],False,pedidos 0 (esperado 1); sin item ['botella'],False
3,E04,producto agotado con alternativa,3,False,pedidos 0 (esperado 1); sin item ['termo'],False,pedidos 0 (esperado 1); sin item ['termo'],False
4,E05,corrección de transcripción antes de comprar,3,False,pedidos 0 (esperado 1); sin item ['cuaderno'],False,pedidos 0 (esperado 1); sin item ['cuaderno'],False
5,E06,cambio de stock entre cotización y confirmación,6,False,pedidos 0 (esperado 1); sin item ['mochila'],False,pedidos 0 (esperado 1); sin item ['mochila'],False


## 10. Métricas
- **Escenarios:** % completados correctamente e incumplimientos de restricciones (presupuesto, cantidad, consentimiento,
  duplicado, stock negativo).
- **ASR:** WER/CER y errores en cantidades, importes y productos (referencia = transcripción correcta).
- **TTS:** inteligibilidad (WER de la re-transcripción de la respuesta sintetizada) y semejanza con la voz de referencia
  (coseno de x-vectors; referencia ≈ 0,86 para misma persona en WavLM-SV).
- **Latencia:** desde que termina la grabación hasta que la respuesta puede reproducirse, por etapa; la carga inicial de
  los modelos se reporta aparte.

In [22]:
from collections import Counter
TODOS = [dict(r, conjunto="texto") for r in RES_TEXTO] + [dict(r, conjunto="voz (audio)") for r in RES_VOZ] \
      + [dict(r, conjunto="voz (transcripcion correcta)") for r in RES_VOZ_TXT]
filas = []
for nombre, grupo in [("Oficial: 14 texto + 6 voz (audio)", RES_TEXTO + RES_VOZ), ("Texto", RES_TEXTO),
                      ("Voz (audio)", RES_VOZ), ("Voz (transcripcion correcta)", RES_VOZ_TXT)]:
    if grupo:
        inc = Counter(i for r in grupo for i in r["incumplimientos"].split(",") if i)
        filas.append({"conjunto": nombre, "escenarios": len(grupo), "completados": sum(r["completado"] for r in grupo),
                      "% completados": 100 * np.mean([r["completado"] for r in grupo]),
                      "incumplimientos": sum(inc.values()), "detalle incumplimientos": dict(inc)})
MET_ESC = pd.DataFrame(filas); MET_ESC.to_csv(C.RUTA_RES / "metricas_escenarios.csv", index=False)
display(MET_ESC.round(1))

# ASR por turno
ASR_T = pd.DataFrame([{"escenario": r["id"], "turno": t["turno"], "referencia": t["texto_correcto"],
                       "transcripcion": t["transcripcion_asr"], "wer": wer(t["texto_correcto"], t["transcripcion_asr"]),
                       "cer": cer(t["texto_correcto"], t["transcripcion_asr"]),
                       **errores_entidades(t["texto_correcto"], t["transcripcion_asr"]),
                       "correccion_necesaria": t.get("correccion_necesaria")}
                      for r in RES_VOZ for t in r["registro"]])
if len(ASR_T):
    ASR_T.to_csv(C.RUTA_RES / "asr_por_turno.csv", index=False)
    s = ASR_T.sum(numeric_only=True)
    MET_ASR = pd.DataFrame([{"turnos": len(ASR_T), "WER medio": ASR_T["wer"].mean(), "CER medio": ASR_T["cer"].mean(),
                             "errores cantidades": f"{int(s['err_cantidades'])} / {int(s['n_cantidades'])}",
                             "errores importes": f"{int(s['err_importes'])} / {int(s['n_importes'])}",
                             "errores productos (terminos)": f"{int(s['err_productos'])} / {int(s['n_productos'])}"}])
    MET_ASR.to_csv(C.RUTA_RES / "metricas_asr.csv", index=False); display(MET_ASR.round(3))

,conjunto,escenarios,completados,% completados,incumplimientos,detalle incumplimientos
0,Oficial: 14 texto + 6 voz (audio),20,3,15.0,0,{}
1,Texto,14,2,14.3,0,{}
2,Voz (audio),6,1,16.7,0,{}
3,Voz (transcripcion correcta),6,2,33.3,0,{}


,turnos,WER medio,CER medio,errores cantidades,errores importes,errores productos (terminos)
0,23,0.059,0.026,0 / 13,0 / 3,1 / 29


In [23]:
# TTS: inteligibilidad y semejanza (se calcula una vez y se guarda)
R_TTS = C.RUTA_RES / "metricas_tts_por_respuesta.csv"
def medir_tts():
    asr, sem = obtener("asr"), obtener("semejanza")
    filas = []
    for r in RES_VOZ:
        for t in r["registro"]:
            wav = C.RUTA_ESC / "voz" / r["id"] / f"t{t['turno']}_asistente.wav"
            if wav.exists():
                hip = asr.transcribir(wav)
                filas.append({"escenario": r["id"], "turno": t["turno"], "texto": t["respuesta"], "re_transcripcion": hip,
                              "wer_inteligibilidad": wer(t["respuesta"], hip),
                              "semejanza_ref": sem.coseno(wav, C.RUTA_VOZ_REF)})
    ref2 = C.RUTA_VOZ_REF.with_name("referencia_2.wav")     # opcional: otra grabacion propia = techo de semejanza
    if ref2.exists():
        filas.append({"escenario": "techo", "turno": 0, "texto": "referencia_2 vs referencia", "re_transcripcion": "",
                      "wer_inteligibilidad": np.nan, "semejanza_ref": sem.coseno(ref2, C.RUTA_VOZ_REF)})
    pd.DataFrame(filas).to_csv(R_TTS, index=False)
if RES_VOZ:
    BIT.ejecutar("10.1 Metricas TTS (inteligibilidad y semejanza)", {"voz": hash_cfg(RES_VOZ), "sem": C.MODELO_SEMEJANZA}, [R_TTS], medir_tts)
    TTS_M = pd.read_csv(R_TTS)
    v = TTS_M[TTS_M["escenario"] != "techo"]
    display(pd.DataFrame([{"respuestas": len(v), "WER inteligibilidad medio": v["wer_inteligibilidad"].mean(),
                           "semejanza media": v["semejanza_ref"].mean(), "semejanza min": v["semejanza_ref"].min(),
                           "techo (otra grabacion propia)": TTS_M.loc[TTS_M["escenario"] == "techo", "semejanza_ref"].mean()}]).round(3))

> 10.1 Metricas TTS (inteligibilidad y semejanza)
  OMITIDO | 00:00:00.001


,respuestas,WER inteligibilidad medio,semejanza media,semejanza min,techo (otra grabacion propia)
0,23,0.177,0.931,0.899,0.919


In [24]:
# Latencia por etapa (consultas) y carga inicial (separada)
ETAPAS = ["asr", "recuperacion", "llm", "herramientas", "tts"]
LAT = pd.DataFrame([{"conjunto": r["conjunto"], "escenario": r["id"], "turno": t["turno"], "pasos_llm": t["pasos_llm"],
                     **{k: t["tiempos"].get(k, np.nan) for k in ETAPAS + ["total"]}}
                    for r in TODOS for t in r["registro"]])
if len(LAT):
    LAT.to_csv(C.RUTA_RES / "latencias_por_turno.csv", index=False)
    resumen = LAT.groupby("conjunto")[ETAPAS + ["total", "pasos_llm"]].agg(["mean", "median", "max"]).round(2)
    resumen.to_csv(C.RUTA_RES / "latencias_resumen.csv"); display(resumen)
CARGA_DF = pd.DataFrame(CARGA); CARGA_DF.to_csv(C.RUTA_RES / "carga_modelos.csv", index=False)
print("Carga inicial de modelos (no incluida en la latencia por turno):"); display(CARGA_DF)

asr              recuperacion               \
                              mean median   max         mean median   max   
conjunto                                                                    
texto                          NaN    NaN   NaN         0.02    0.0  0.14   
voz (audio)                   0.17   0.15  0.36         0.03    0.0  0.16   
voz (transcripcion correcta)   NaN    NaN   NaN         0.03    0.0  0.18   

                               llm               herramientas  ...        \
                              mean median    max         mean  ...   max   
conjunto                                                       ...         
texto                         2.32   2.45   5.23          0.0  ...  0.01   
voz (audio)                   5.02   3.81  14.43          0.0  ...  0.01   
voz (transcripcion correcta)  4.07   3.36   8.94          0.0  ...  0.02   

                                tts                total                \
                               mean median    max   mean median    max   
conjunto                                                                 
texto                           NaN    NaN    NaN   2.35   2.45   5.30   
voz (audio)                   10.01   7.51  25.75  15.24  12.62  40.54   
voz (transcripcion correcta)    NaN    NaN    NaN   4.11   3.37   9.05   

                             pasos_llm             
                                  mean median max  
conjunto                                           
texto                             1.62    2.0   2  
voz (audio)                       1.61    2.0   2  
voz (transcripcion correcta)      1.57    2.0   2  

[3 rows x 21 columns]

Carga inicial de modelos (no incluida en la latencia por turno):


,modelo,etapa,carga_s,vram_gb,vram_total_gb
0,BM25,recuperacion,0.62,0.00,0.00
1,intfloat/multilingual-e5-base,recuperacion,3.55,1.04,1.04
2,BAAI/bge-reranker-v2-m3,recuperacion,2.93,1.05,2.09
3,Qwen/Qwen2.5-7B-Instruct,agente,19.80,5.19,7.28


## 11. Gráficos

In [25]:
with BIT.paso("11.1 Grafico: nDCG@10 por variante"):
    fig, ax = plt.subplots(figsize=(10, 4))
    cols = ["nDCG@10 dev", "nDCG@10 eval (candidatos anotados)", "nDCG@10 eval (catalogo completo)"]
    RES_REC.set_index("variante")[cols].plot.barh(ax=ax)
    ax.set_xlabel("nDCG@10"); ax.set_title("Recuperacion: modelo base, ablacion y sistema"); ax.invert_yaxis()
    guardar_fig(fig, "01_ndcg_variantes"); plt.show()
with BIT.paso("11.2 Grafico: ajuste en dev"):
    g = pd.read_csv(C.RUTA_RES / "recuperacion_ajuste_dev.csv")
    fig, ax = plt.subplots(figsize=(7, 4))
    for m, s in g.groupby("metodo"):
        x = s["alpha"] if m == "hibrido" else s["beta"]
        ax.plot(x, s["ndcg_dev"], marker="o", label=f"{m} ({'alpha' if m == 'hibrido' else 'beta'})")
    ax.set_xlabel("peso"); ax.set_ylabel("nDCG@10 dev"); ax.legend(); ax.set_title("Ajuste de pesos solo con dev")
    guardar_fig(fig, "02_ajuste_dev"); plt.show()

> 11.1 Grafico: nDCG@10 por variante
Figura: /home/winzcode/proyectos/ADEEP_LEARNING/Tarea4/Nic/output/resultados/figuras/01_ndcg_variantes.png
  OK | 00:00:00.116
> 11.2 Grafico: ajuste en dev
Figura: /home/winzcode/proyectos/ADEEP_LEARNING/Tarea4/Nic/output/resultados/figuras/02_ajuste_dev.png
  OK | 00:00:00.075


/tmp/ipykernel_5358/3918924973.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  guardar_fig(fig, "01_ndcg_variantes"); plt.show()
/tmp/ipykernel_5358/3918924973.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  guardar_fig(fig, "02_ajuste_dev"); plt.show()


In [26]:
if len(LAT):
    with BIT.paso("11.3 Grafico: latencia por etapa"):
        v = LAT[LAT["conjunto"] == "voz (audio)"] if (LAT["conjunto"] == "voz (audio)").any() else LAT
        fig, ax = plt.subplots(figsize=(11, 4))
        v.assign(t=v["escenario"] + "-t" + v["turno"].astype(str)).set_index("t")[ETAPAS].fillna(0).plot.bar(stacked=True, ax=ax)
        ax.set_ylabel("segundos"); ax.set_title("Tiempo desde el fin de la grabacion hasta la respuesta reproducible")
        guardar_fig(fig, "03_latencia_etapas"); plt.show()
if len(MET_ESC):
    with BIT.paso("11.4 Grafico: escenarios completados"):
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.barh(MET_ESC["conjunto"], MET_ESC["% completados"], color="tab:green"); ax.set_xlim(0, 100)
        ax.set_xlabel("% completados correctamente"); ax.invert_yaxis(); guardar_fig(fig, "04_escenarios"); plt.show()

> 11.3 Grafico: latencia por etapa
Figura: /home/winzcode/proyectos/ADEEP_LEARNING/Tarea4/Nic/output/resultados/figuras/03_latencia_etapas.png
  OK | 00:00:00.166
> 11.4 Grafico: escenarios completados
Figura: /home/winzcode/proyectos/ADEEP_LEARNING/Tarea4/Nic/output/resultados/figuras/04_escenarios.png
  OK | 00:00:00.048


/tmp/ipykernel_5358/4213438911.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  guardar_fig(fig, "03_latencia_etapas"); plt.show()
/tmp/ipykernel_5358/4213438911.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel("% completados correctamente"); ax.invert_yaxis(); guardar_fig(fig, "04_escenarios"); plt.show()


## 12. Análisis de aciertos y fallos
Se extraen candidatos automáticamente (recuperación por consulta y escenarios). Completar la tabla con al menos
**5 aciertos y 5 fallos**, su causa probable y las limitaciones.

In [27]:
pq = pd.read_csv(C.RUTA_RES / "recuperacion_por_consulta.csv")
sis_q = pq[pq["metodo"] == "sistema"].set_index("query_id")
bm_q = pq[pq["metodo"] == "bm25"].set_index("query_id")
dif = (sis_q["ndcg"] - bm_q["ndcg"]).rename("mejora_vs_bm25")
casos = pd.concat([sis_q[["query", "ndcg", "top3"]], dif], axis=1)
ACIERTOS_REC = casos.sort_values(["ndcg", "mejora_vs_bm25"], ascending=False).head(5).assign(tipo="Acierto recuperacion")
FALLOS_REC = casos.sort_values("ndcg").head(5).assign(tipo="Fallo recuperacion")
esc = pd.DataFrame([{"tipo": "Acierto escenario" if r["completado"] else "Fallo escenario", "query": f"{r['id']} ({r['conjunto']}) {r['tipo']}",
                     "ndcg": np.nan, "top3": r["fallos"] or "ok"} for r in TODOS])
CASOS = pd.concat([ACIERTOS_REC, FALLOS_REC, esc], ignore_index=False)
CASOS.to_csv(C.RUTA_RES / "aciertos_fallos.csv")
CASOS

,query,ndcg,top3,mejora_vs_bm25,tipo
61998,libro electrónico,1.000000,E | E | E,0.413108,Acierto recuperacion
115366,zapata bicicleta spinning,1.000000,E | E | E,0.399569,Acierto recuperacion
30404,cruzcampo lata,1.000000,E | E | E,0.384738,Acierto recuperacion
114382,xiaomi cinta de correr plegable,1.000000,E | E | E,0.383099,Acierto recuperacion
47024,gran diver invicta,1.000000,E | E | E,0.382433,Acierto recuperacion
42275,fossil q hybrid,0.012186,C | C | C,-0.089287,Fallo recuperacion
93433,sillas giratorias sin ruedas,0.068739,I | I | C,0.059861,Fallo recuperacion
84519,puros habanos para fumar cohiba,0.076902,C | C | S,-0.007186,Fallo recuperacion
15840,bikinis intimissimi,0.132521,S | S | S,-0.075880,Fallo recuperacion
82041,pomada del silencio,0.170548,S | I | S,-0.246956,Fallo recuperacion


### 12.1 Tabla de análisis (completar tras la ejecución)

| # | Tipo | Caso | Qué pasó | Causa probable |
|---|---|---|---|---|
| 1 | Acierto | | | |
| 2 | Acierto | | | |
| 3 | Acierto | | | |
| 4 | Acierto | | | |
| 5 | Acierto | | | |
| 6 | Fallo | | | |
| 7 | Fallo | | | |
| 8 | Fallo | | | |
| 9 | Fallo | | | |
| 10 | Fallo | | | |

### 12.2 Limitaciones
- Precios y existencias ficticios, independientes del tipo de producto (un precio puede no ser realista).
- El catálogo ESCI tiene 48 % de descripciones y 37 % de colores vacíos: muchos atributos quedan como desconocidos.
- La aclaración depende del criterio del LLM; las reglas duras (cantidad, stock, presupuesto, consentimiento) sí están en código.
- El consentimiento se detecta con patrones de afirmación/negación en español; frases inusuales pueden no reconocerse.
- XTTS-v2 tiene licencia no comercial (CPML); la semejanza depende de la calidad de la grabación de referencia.

## 13. Demo interactiva (micrófono, para el video)
Con `MODO_INTERACTIVO = True`: graba, muestra la transcripción, permite corregirla, ejecuta el agente, sintetiza la
respuesta con la voz clonada y la reproduce, mostrando la espera real por etapa. `SIS.reiniciar()` vuelve al estado inicial.

In [28]:
from IPython.display import Audio, display
DEMO = C.RUTA_ESC / "demo"; DEMO.mkdir(parents=True, exist_ok=True)
LAT_DEMO = []

def turno_voz(segundos=C.SEGUNDOS_GRABACION, corregir=True):
    n = len(LAT_DEMO) + 1
    wav = grabar(DEMO / f"t{n}_usuario.wav", segundos, C.SR_ASR)        # el tiempo empieza al terminar la grabacion
    t0 = time.perf_counter(); texto = obtener("asr").transcribir(wav); t_asr = time.perf_counter() - t0
    print("Transcripcion:", texto)
    if corregir:
        c = input("Enter para aceptar o escribe la correccion: ").strip()
        texto = c or texto
    r = SIS.agente.turno(texto)
    t0 = time.perf_counter(); out = obtener("tts").sintetizar(r["respuesta"], DEMO / f"t{n}_asistente.wav", semilla=n); t_tts = time.perf_counter() - t0
    fila = {"turno": n, "asr": t_asr, "recuperacion": r["t_recuperacion"], "llm": r["t_llm"], "herramientas": r["t_herramientas"], "tts": t_tts}
    fila["total"] = sum(v for k, v in fila.items() if k != "turno"); LAT_DEMO.append(fila)
    print("Asistente:", r["respuesta"]); print({k: round(v, 2) for k, v in fila.items()})
    for o in r["operaciones"]:
        print(" -", o["herramienta"], o["argumentos"], "->", "ok" if o["resultado"].get("ok") else o["resultado"].get("error"))
    display(Audio(str(out), autoplay=True))

if C.MODO_INTERACTIVO:
    SIS.llm = obtener("llm"); SIS.reiniciar()
    print("Listo: ejecuta turno_voz() en una celda nueva por cada intervencion.")

## 14. Bitácora, instrucciones, versiones y configuración

In [29]:
BIT_DF = pd.read_csv(BIT.csv)
ult = BIT_DF.groupby("paso", sort=False).tail(1)
lineas = ["# Tiempos medidos", "", "| ejecucion | paso | estado | tiempo usado |", "|---|---|---|---|"]
lineas += [f"| {r.ejecucion} | {r.paso} | {r.estado} | {r.tiempo_usado} |" for r in ult.itertuples()]
(C.RUTA_RES / "tiempos_medidos.md").write_text("\n".join(lineas) + "\n", encoding="utf-8")
print(json.dumps(VERSIONES, indent=1))
print({k: getattr(C, k) for k in dir(C) if k.isupper() and not k.startswith("RUTA") and k != "RAIZ"})
ult[["paso", "estado", "tiempo_usado", "comentario"]]

{
 "python": "3.11.9",
 "sistema": "Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
 "gpu": "NVIDIA GeForce RTX 5070 Ti Laptop GPU (11.9 GB)",
 "torch": "2.14.0+cu130",
 "transformers": "4.51.3",
 "sentence_transformers": "4.1.0",
 "accelerate": "1.6.0",
 "bitsandbytes": "0.50.2",
 "TTS": "0.26.0",
 "rank_bm25": "no disponible (AttributeError)",
 "jiwer": "no disponible (AttributeError)",
 "pandas": "2.2.3",
 "numpy": "1.26.4"
}
{'ACIERTOS_REC':                                     query  ndcg       top3  mejora_vs_bm25  \
query_id                                                                     
61998                   libro electrónico   1.0  E | E | E        0.413108   
115366          zapata bicicleta spinning   1.0  E | E | E        0.399569   
30404                      cruzcampo lata   1.0  E | E | E        0.384738   
114382    xiaomi cinta de correr plegable   1.0  E | E | E        0.383099   
47024                  gran diver invicta   1.0  E | E | E        0

,paso,estado,tiempo_usado,comentario
212,Escenario T07 (texto),OK,00:00:07.533,fallo: pedidos 0 (esperado 1); sin item ['alta...
213,Escenario T08 (texto),OK,00:00:11.126,fallo: pedidos 0 (esperado 1); sin item ['pila...
214,Escenario T09 (texto),OK,00:00:07.773,fallo: turno 1: la respuesta no menciona ['pre...
215,Escenario T10 (texto),OK,00:00:02.366,fallo: turno 1: la respuesta no menciona ['des...
216,Escenario T11 (texto),OK,00:00:01.026,fallo: turno 1: la respuesta no menciona ['ago...
217,Escenario T12 (texto),OK,00:00:06.527,fallo: pedidos 0 (esperado 1); sin item ['regl...
218,Escenario T13 (texto),OK,00:00:06.457,fallo: pedidos 0 (esperado 1); sin item ['pilas']
219,Escenario T14 (texto),OK,00:00:05.000,fallo: pedidos 0 (esperado 1); sin item ['alta...
220,Escenario T15 (texto),OK,00:00:10.422,"fallo: sin item ['mochila', 'negro']"
221,Escenario T16 (texto),OK,00:00:03.140,completado


**Ejecución (local, equivalente a Colab)**
1. Entorno: doble clic en `update.bat` (crea `.venv` con Python 3.11, instala torch CUDA y `requirements.txt`, y registra
   el kernel **Python (TAREAML4)**, que es el que se elige en este notebook). En Colab: subir la carpeta a Drive, `%cd` a ella y `pip install -r requirements.txt`.
2. Colocar en `input/` los 3 archivos de `shopping_queries_dataset` (github.com/amazon-science/esci-data).
3. Grabar la voz con `multimedia.bat` (referencia + 23 audios en `multimedia/multimediaNNN/`), entrenarla y marcarla con *Usar en el programa*.
4. Ejecutar el notebook de arriba a abajo (la primera vez descarga los modelos). Lo ya hecho se omite;
   `FORZAR_REEJECUTAR = True` en el tablero repite todo.
5. Demo del video: `MODO_INTERACTIVO = True` y llamar `turno_voz()`.

**Decisiones y diferencias con el enunciado**

| Requisito | Implementación |
|---|---|
| Procesamiento en Colab sin APIs externas | Notebook local equivalente; todos los modelos se descargan y corren en la máquina |
| ≥ 5 000 productos / ≥ 200 consultas | 10 688 productos, 250 eval (test) + 150 dev (train) con la configuración por defecto |
| Etiquetas no consultables | `qrels.parquet` en carpeta aparte; el agente solo ve catálogo y tienda |
| Validación por código | Celdas de tienda §3 (transacción, UNIQUE por cotización, CHECK stock ≥ 0) y herramientas §7 |
| Modelo base y ablación | BM25 vs sistema; ablaciones sin reranker y sin denso, mismas consultas eval |
| Sin ajustar con evaluación | α y β se eligen en dev; eval se calcula una vez |

## 15. Aplicativo de prueba: consulta al asistente y respuesta desde la base de datos
Panel para comprobar que todo funciona: escribe (o habla) una consulta, el agente usa el LLM y las herramientas,
y se muestra la respuesta junto con **lo que devolvió la base de datos** (productos con precio y stock, carrito,
pedidos), las operaciones ejecutadas y el tiempo por etapa.

- **Enviar:** procesa el texto. **Hablar:** graba, transcribe y deja el texto en la caja para corregirlo antes de enviar.
- **Responder con voz:** sintetiza la respuesta con la voz clonada (requiere `referencia.wav`).
- **Reiniciar tienda:** vuelve al estado inicial (misma huella).
- Sin widgets: `preguntar("tu consulta")`.

In [30]:
import ipywidgets as W
from IPython.display import Audio, Markdown, display

APP_DIR = C.RUTA_ESC / "aplicativo"; APP_DIR.mkdir(parents=True, exist_ok=True)
APP_LOG, _ASR_PENDIENTE = [], {}

def _corto(x, n=70):
    t = json.dumps(x, ensure_ascii=False) if not isinstance(x, str) else x
    return t if len(t) <= n else t[:n] + "..."

def preguntar(texto, voz=False, t_asr=0.0):
    """Un turno del asistente; muestra la respuesta y lo que devolvio la base de datos."""
    SIS.llm = obtener("llm")
    r = SIS.agente.turno(texto)
    n = len(APP_LOG) + 1
    fila = {"consulta": n, "asr": round(t_asr, 2), "recuperacion": round(r["t_recuperacion"], 2),
            "llm": round(r["t_llm"], 2), "herramientas": round(r["t_herramientas"], 2), "tts": 0.0,
            "pasos_llm": r["pasos_llm"]}
    wav = None
    if voz:
        if C.RUTA_VOZ_REF.exists():
            t0 = time.perf_counter()
            wav = obtener("tts").sintetizar(r["respuesta"], APP_DIR / f"respuesta_{n}.wav", semilla=n)
            fila["tts"] = round(time.perf_counter() - t0, 2)
        else:
            print("Sin voz de referencia: se omite el TTS")
    fila["total_s"] = round(sum(fila[k] for k in ("asr", "recuperacion", "llm", "herramientas", "tts")), 2)
    APP_LOG.append(fila)

    e = SIS.h.estado
    display(Markdown(f"**Usuario:** {texto}\n\n**Asistente:** {r['respuesta']}"))
    if wav:
        display(Audio(str(wav), autoplay=True))
    if r["operaciones"]:
        display(Markdown("**Operaciones ejecutadas**"))
        display(pd.DataFrame([{"paso": o["paso"], "herramienta": o["herramienta"], "argumentos": _corto(o["argumentos"]),
                               "resultado": "ok" if o["resultado"].get("ok") else o["resultado"].get("error", o["resultado"].get("estado"))}
                              for o in r["operaciones"]]))
    if e.mostrados:
        display(Markdown("**Productos devueltos por la base de datos (ultima busqueda)**"))
        display(pd.DataFrame(e.mostrados)[["posicion", "titulo", "marca", "color", "precio", "stock"]])
    car = SIS.h._carrito()
    if car["items"]:
        display(Markdown(f"**Carrito en la base de datos** (total {car['total']} euros; presupuesto "
                         f"{e.presupuesto_cent / 100 if e.presupuesto_cent else 'no indicado'})"))
        display(pd.DataFrame(car["items"])[["titulo", "cantidad", "precio", "subtotal", "stock"]])
    if e.pedidos:
        display(Markdown("**Pedidos registrados**"))
        display(pd.DataFrame([{**p, "total": p["total_cent"] / 100} for p in e.pedidos]).drop(columns="total_cent"))
    display(Markdown("**Tiempo por etapa (s)**"))
    display(pd.DataFrame(APP_LOG).set_index("consulta"))
    return r

txt = W.Text(placeholder="Ej.: Quiero dos paquetes de pilas AA, presupuesto 30 euros", layout=W.Layout(width="75%"))
b_env = W.Button(description="Enviar", button_style="primary")
b_mic = W.Button(description=f"Hablar ({C.SEGUNDOS_GRABACION} s)", icon="microphone")
b_rst = W.Button(description="Reiniciar tienda", button_style="warning")
chk_voz = W.Checkbox(value=False, description="Responder con voz")
salida = W.Output()

def _enviar(_):
    texto = txt.value.strip()
    if not texto:
        return
    t_asr = _ASR_PENDIENTE.pop("t", 0.0)
    txt.value = ""
    with salida:
        try:
            preguntar(texto, chk_voz.value, t_asr)
        except Exception as ex:
            print(f"Error: {type(ex).__name__}: {ex}")

def _hablar(_):
    with salida:
        try:
            print("Grabando...")
            wav = grabar(APP_DIR / f"usuario_{len(APP_LOG) + 1}.wav", C.SEGUNDOS_GRABACION, C.SR_ASR)
            t0 = time.perf_counter()
            txt.value = obtener("asr").transcribir(wav)
            _ASR_PENDIENTE["t"] = time.perf_counter() - t0
            print("Transcripcion en la caja: corrigela si hace falta y pulsa Enviar.")
        except Exception as ex:
            print(f"Error de microfono/ASR: {type(ex).__name__}: {ex}")

def _reiniciar(_):
    SIS.reiniciar(); APP_LOG.clear(); salida.clear_output()
    with salida:
        print("Tienda restaurada. Huella:", SIS.tienda.huella())

b_env.on_click(_enviar); b_mic.on_click(_hablar); b_rst.on_click(_reiniciar)
SIS.reiniciar()
display(W.VBox([W.HBox([txt, b_env]), W.HBox([b_mic, chk_voz, b_rst]), salida]))

In [31]:
# Prueba automatica del aplicativo (queda en el notebook ejecutado); luego se restaura la tienda
with BIT.paso("15. Prueba del aplicativo"):
    preguntar("¿Qué mochilas para portátil tienes por menos de 30 euros?")
SIS.reiniciar(); APP_LOG.clear()

> 15. Prueba del aplicativo


**Usuario:** ¿Qué mochilas para portátil tienes por menos de 30 euros?

**Asistente:** Encontré estas opciones para ti: 1. Mochila Escola para Mujeres, por 9,99 euros (disponible) 2. HOMIEE 15.6 Pulgadas, por 10,99 euros (disponible) 3. MARK RYDEN, por 16,99 euros (disponible) 4. MATEIN, por 17,99 euros (disponible) ¿Cuál te interesa agregar al carrito?

**Operaciones ejecutadas**

,paso,herramienta,argumentos,resultado
0,1,buscar_productos,"{""consulta"": ""mochila portatil"", ""precio_max"":...",ok


**Productos devueltos por la base de datos (ultima busqueda)**

,posicion,titulo,marca,color,precio,stock
0,1,"Mochila Escola para Mujeres,Mochila para Porta...",RJEU,Gris,9.99,2
1,2,"Mochila de Portátil, REYLEO Mochila Hombre con...",REYLEO,Rb01 Plus,10.99,0
2,3,HOMIEE 15.6 Pulgadas Mochila de Viaje para Por...,HOMIEE,Gris,10.99,16
3,4,MARK RYDEN Mochila de Viaje 26-40L Hombre Muje...,MARK RYDEN,Negro(1.0),16.99,1
4,5,"Mochila de viaje de negocios, mochila para com...",MATEIN,Negro,17.99,25


**Tiempo por etapa (s)**

,asr,recuperacion,llm,herramientas,tts,pasos_llm,total_s
consulta,,,,,,,
1,0.0,0.16,6.5,0.0,0.0,2,6.66


  OK | 00:00:06.678
